In [7]:
# Phase 1 - Cell 1: Always run this first
import os, sys, importlib

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
MUSDB_PATH   = "D:/dataset"

# Clear stale cache
for k in list(sys.modules.keys()):
    if "src" in k:
        del sys.modules[k]

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("done")

Device : cuda
PyTorch: 2.1.0+cu118
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM   : 6.4 GB
done


In [2]:
# Phase 1 - Cell 2: Write src/models/unet.py
# This is the main neural network for Phase 1

import os

unet_code = '''"""
src/models/unet.py
==================
Baseline Magnitude U-Net for source separation.

WHAT IS A U-NET?
----------------
A U-Net has two halves:

  ENCODER (left side) - reads the spectrogram, extracts features
  Each encoder block:  Doubles channels, halves the time dimension
  [B, 1,  F, T]
     ↓  ConvBlock(1→32)
  [B, 32, F, T]   ← skip connection saved here
     ↓  ConvBlock(32→64) + downsample
  [B, 64, F, T//2] ← skip connection saved here
     ↓  ... and so on

  BOTTLENECK (middle) - deepest understanding of the audio
  [B, 256, F, T//8]

  DECODER (right side) - reconstructs the mask
  Each decoder block: Halves channels, doubles time dimension
  Uses skip connections from encoder (the "U" shape!)
  [B, 256, F, T//8]
     ↑  ConvBlock + upsample + skip from encoder
  [B, 128, F, T//4]
     ↑  ... and so on
  [B, 1, F, T]   ← vocal mask

WHY SKIP CONNECTIONS?
---------------------
The encoder loses fine details as it compresses.
Skip connections let the decoder "look back" at the original
encoder features to recover those details.
Think of it like: decoder asks encoder "hey what did you see
at this scale?" and the encoder replies with its memory.

MASK ESTIMATION:
----------------
Output is a mask in [0, 1].
  0 = silence (remove this frequency at this time)
  1 = keep (this is definitely vocal)
  0.5 = uncertain

Final output: vocals = mixture_magnitude * mask
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Tuple, Optional


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class UNetConfig:
    """
    All U-Net hyperparameters.

    n_freq_bins : number of frequency bins from STFT (n_fft//2+1 = 1025)
    base_channels : channels in first encoder layer (doubles each layer)
    depth : number of encoder/decoder layers
    dropout : dropout probability (regularisation)
    """
    n_freq_bins:   int   = 1025
    base_channels: int   = 32       # → 32, 64, 128, 256
    depth:         int   = 4
    dropout:       float = 0.1


# =============================================================================
# BUILDING BLOCK: ConvBlock
# =============================================================================

class ConvBlock(nn.Module):
    """
    Two Conv2d layers with BatchNorm and LeakyReLU.

    Think of this as the basic "processing unit" of our network.
    Two convolutions let the block learn more complex patterns
    than a single convolution.

    Shape: [B, C_in, F, T] → [B, C_out, F, T]   (same spatial size)
    """

    def __init__(
        self,
        in_channels:  int,
        out_channels: int,
        dropout:      float = 0.1,
    ):
        super().__init__()

        self.block = nn.Sequential(
            # First convolution
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout),

            # Second convolution
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C_in, F, T]
        return self.block(x)
        # output: [B, C_out, F, T]


# =============================================================================
# ENCODER BLOCK
# =============================================================================

class EncoderBlock(nn.Module):
    """
    One step of the encoder:
      1. ConvBlock  (learn features)
      2. MaxPool2d  (halve the time dimension → compress)

    Returns BOTH the features (for skip connection) AND the
    downsampled version (to pass to next encoder).

    Input  shape: [B, C_in,  F, T]
    Output shape: [B, C_out, F, T//2]  (downsampled)
    Skip   shape: [B, C_out, F, T]     (before downsampling)
    """

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.1):
        super().__init__()
        self.conv    = ConvBlock(in_channels, out_channels, dropout)
        # Pool only along time axis (dim=3), keep frequency axis (dim=2) intact
        # Why? Frequency structure is musically meaningful - don't compress it
        self.pool    = nn.MaxPool2d(kernel_size=(1, 2))  # (freq, time)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        features    = self.conv(x)          # [B, C_out, F, T]
        downsampled = self.pool(features)   # [B, C_out, F, T//2]
        return downsampled, features        # return both!


# =============================================================================
# DECODER BLOCK
# =============================================================================

class DecoderBlock(nn.Module):
    """
    One step of the decoder:
      1. Upsample  (double the time dimension → expand)
      2. Concatenate skip connection from encoder
      3. ConvBlock (learn to combine encoder + decoder features)

    The skip connection cat doubles the channel count temporarily,
    so in_channels here = decoder_channels + skip_channels.

    Input  shape: [B, C_in, F, T]     (from previous decoder layer)
    Skip   shape: [B, C_skip, F, T*2] (from encoder)
    Output shape: [B, C_out, F, T*2]  (upsampled)
    """

    def __init__(
        self,
        in_channels:   int,
        skip_channels: int,
        out_channels:  int,
        dropout:       float = 0.1,
    ):
        super().__init__()
        # After concat: in_channels + skip_channels
        self.conv = ConvBlock(in_channels + skip_channels, out_channels, dropout)

    def forward(
        self,
        x:    torch.Tensor,
        skip: torch.Tensor,
    ) -> torch.Tensor:
        # 1. Upsample time dimension by 2
        x = F.interpolate(x, scale_factor=(1, 2), mode="bilinear", align_corners=False)
        # x: [B, C_in, F, T*2]

        # 2. Match size to skip (in case of off-by-one from odd dimensions)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)

        # 3. Concatenate along channel dimension
        x = torch.cat([x, skip], dim=1)
        # x: [B, C_in + C_skip, F, T*2]

        # 4. ConvBlock to fuse
        return self.conv(x)
        # output: [B, C_out, F, T*2]


# =============================================================================
# MAIN U-NET
# =============================================================================

class MagnitudeUNet(nn.Module):
    """
    Full U-Net for magnitude spectrogram source separation.

    Input : mixture magnitude  [B, 1, F, T]
            (single channel - we process left/right separately or use mono)
    Output: soft mask          [B, 1, F, T]  values in [0, 1]

    Usage:
        model  = MagnitudeUNet(config)
        mask   = model(mixture_magnitude)
        vocals = mixture_magnitude * mask
    """

    def __init__(self, config: UNetConfig):
        super().__init__()
        self.config = config

        # Channel sizes at each depth level
        # depth=4 → [32, 64, 128, 256]
        channels = [config.base_channels * (2 ** i) for i in range(config.depth)]
        # channels = [32, 64, 128, 256]

        # ── Encoder ──────────────────────────────────────────────────
        # First encoder: 1 input channel → base_channels
        self.encoders = nn.ModuleList()

        self.encoders.append(
            EncoderBlock(1, channels[0], config.dropout)
        )
        # Remaining encoders: channels[i] → channels[i+1]
        for i in range(1, config.depth):
            self.encoders.append(
                EncoderBlock(channels[i-1], channels[i], config.dropout)
            )
        # encoders[0]: 1   → 32   (skip saved, output downsampled)
        # encoders[1]: 32  → 64
        # encoders[2]: 64  → 128
        # encoders[3]: 128 → 256

        # ── Bottleneck ───────────────────────────────────────────────
        # Deepest layer - no skip connection, no pooling
        self.bottleneck = ConvBlock(channels[-1], channels[-1], config.dropout)
        # 256 → 256

        # ── Decoder ──────────────────────────────────────────────────
        self.decoders = nn.ModuleList()
        # Build in reverse: decoder[0] handles deepest skip
        for i in range(config.depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(
                    in_channels   = channels[i],
                    skip_channels = channels[i],   # skip has same channels as encoder output
                    out_channels  = channels[i-1],
                    dropout       = config.dropout,
                )
            )
        # Final decoder: back to base_channels
        self.decoders.append(
            DecoderBlock(
                in_channels   = channels[0],
                skip_channels = channels[0],
                out_channels  = channels[0],
                dropout       = config.dropout,
            )
        )
        # decoders[0]: 256+256 → 128
        # decoders[1]: 128+128 → 64
        # decoders[2]: 64+64   → 32
        # decoders[3]: 32+32   → 32

        # ── Output Head ──────────────────────────────────────────────
        # 1x1 conv to collapse channels → 1 channel mask
        self.output_conv = nn.Conv2d(channels[0], 1, kernel_size=1)
        # Sigmoid squashes output to [0, 1] → valid mask range
        self.sigmoid = nn.Sigmoid()

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : mixture magnitude [B, 1, F, T]

        Returns:
            mask : [B, 1, F, T]  soft mask in [0, 1]
        """
        assert x.dim() == 4,  f"Expected [B,1,F,T], got {x.shape}"
        assert x.shape[1] == 1, f"Expected 1 input channel, got {x.shape[1]}"

        # ── Encode ───────────────────────────────────────────────────
        skips = []
        for encoder in self.encoders:
            x, skip = encoder(x)
            skips.append(skip)
        # x    : [B, 256, F, T//16]   (downsampled 2^4 = 16× in time)
        # skips: list of [B,32,F,T], [B,64,F,T//2], [B,128,F,T//4], [B,256,F,T//8]

        # ── Bottleneck ───────────────────────────────────────────────
        x = self.bottleneck(x)
        # x: [B, 256, F, T//16]

        # ── Decode ───────────────────────────────────────────────────
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = decoder(x, skip)
        # x: [B, 32, F, T]

        # ── Output mask ──────────────────────────────────────────────
        x = self.output_conv(x)   # [B, 1, F, T]
        x = self.sigmoid(x)       # [B, 1, F, T]  values in [0,1]

        return x

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        """Returns total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ------------------------------------------------------------------
    def separate(
        self,
        mixture_magnitude: torch.Tensor,
        mixture_phase:     torch.Tensor,
        processor,                         # STFTProcessor instance
        original_length:   int,
    ) -> torch.Tensor:
        """
        Full separation pipeline:
        magnitude + phase  →  separated audio waveform

        Args:
            mixture_magnitude : [B, C, F, T]  (C=2 for stereo)
            mixture_phase     : [B, C, F, T]
            processor         : STFTProcessor
            original_length   : int  (samples, for exact-length iSTFT)

        Returns:
            audio : [B, C, T]  separated waveform
        """
        B, C, F, T = mixture_magnitude.shape

        # Process each channel independently
        masks = []
        for ch in range(C):
            ch_mag  = mixture_magnitude[:, ch:ch+1, :, :]  # [B, 1, F, T]
            ch_mask = self.forward(ch_mag)                  # [B, 1, F, T]
            masks.append(ch_mask)

        # Stack channels back: [B, C, F, T]
        mask = torch.cat(masks, dim=1)                      # [B, C, F, T]

        # Apply mask
        separated_mag = mixture_magnitude * mask            # [B, C, F, T]

        # Reconstruct audio
        stft      = processor.magnitude_phase_to_complex(separated_mag, mixture_phase)
        audio_out = processor.istft(stft, length=original_length)

        return audio_out  # [B, C, T]
'''

path = os.path.join(PROJECT_ROOT, "src", "models", "unet.py")
with open(path, 'w', encoding='utf-8') as f:
    f.write(unet_code)

print(f"✓ Written: {path}")
print(f"  Size: {os.path.getsize(path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\unet.py
  Size: 13,992 bytes


In [3]:
# Phase 1 - Cell 3: Write src/training/losses.py

import os

losses_code = '''"""
src/training/losses.py
=======================
Loss functions for audio source separation.

WHAT IS A LOSS FUNCTION?
-------------------------
It measures "how wrong" the model is.
During training, PyTorch automatically adjusts the model\'s
weights to make the loss smaller.

Loss = 0.0  →  perfect prediction
Loss = big  →  terrible prediction

LOSSES WE USE IN PHASE 1:
--------------------------
1. L1 Loss (Mean Absolute Error)
   - Simple: average absolute difference between prediction and target
   - MAE(pred, target) = mean(|pred - target|)
   - Good for spectrograms: directly penalises wrong magnitudes

2. SI-SNR (Scale-Invariant Signal-to-Noise Ratio)  
   - Measures audio quality in dB
   - Higher is better (positive = good, negative = bad)
   - "Scale-invariant" means volume differences don\'t hurt the score
   - Used as a METRIC (we report this to know how good we are)
   - Also used as a loss: loss = -SI_SNR (minimise negative = maximise SI-SNR)

PHASE 4 will add:
   - Spectral Convergence Loss
   - Log Magnitude Loss  
   - Combined multi-domain loss
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict


# =============================================================================
# L1 MAGNITUDE LOSS
# =============================================================================

class L1MagnitudeLoss(nn.Module):
    """
    L1 loss on magnitude spectrograms.

    Simple and effective for Phase 1.
    Compares predicted magnitude vs target magnitude directly.

    loss = mean( |predicted_magnitude - target_magnitude| )
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        pred_magnitude:   torch.Tensor,   # [B, C, F, T]
        target_magnitude: torch.Tensor,   # [B, C, F, T]
    ) -> torch.Tensor:
        assert pred_magnitude.shape == target_magnitude.shape, (
            f"Shape mismatch: {pred_magnitude.shape} vs {target_magnitude.shape}"
        )
        return F.l1_loss(pred_magnitude, target_magnitude)


# =============================================================================
# SI-SNR LOSS
# =============================================================================

class SISNRLoss(nn.Module):
    """
    Scale-Invariant Signal-to-Noise Ratio loss.

    SI-SNR measures separation quality in decibels (dB).

    HOW IT WORKS:
    1. Project the estimate onto the target  (signal part)
    2. Compute the noise = estimate - signal_part
    3. SI-SNR = 10 * log10( ||signal||² / ||noise||² )

    Higher SI-SNR = better separation.
    As a LOSS we return -SI-SNR (so minimising loss = maximising quality).

    Typical values:
        < 0 dB   : worse than random
        0-5 dB   : poor (but audible separation)
        5-10 dB  : decent separation
        > 10 dB  : good separation
        > 15 dB  : excellent
    """

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(
        self,
        estimate: torch.Tensor,   # [B, C, T]  predicted audio
        target:   torch.Tensor,   # [B, C, T]  ground truth audio
    ) -> torch.Tensor:
        """
        Returns -SI-SNR (negative, because we want to MINIMISE loss).
        """
        assert estimate.shape == target.shape, (
            f"Shape mismatch: {estimate.shape} vs {target.shape}"
        )

        # Flatten channels and time: [B, C, T] → [B, C*T]
        B, C, T   = estimate.shape
        est_flat  = estimate.reshape(B, -1)   # [B, C*T]
        tgt_flat  = target.reshape(B, -1)     # [B, C*T]

        # 1. Zero-mean (remove DC offset)
        est_flat  = est_flat - est_flat.mean(dim=1, keepdim=True)
        tgt_flat  = tgt_flat - tgt_flat.mean(dim=1, keepdim=True)

        # 2. Scale-invariant projection
        # s_target = <est, tgt> / <tgt, tgt> * tgt
        dot       = (est_flat * tgt_flat).sum(dim=1, keepdim=True)          # [B, 1]
        tgt_power = (tgt_flat * tgt_flat).sum(dim=1, keepdim=True) + self.eps
        s_target  = dot / tgt_power * tgt_flat                              # [B, C*T]

        # 3. Noise = residual
        e_noise   = est_flat - s_target                                      # [B, C*T]

        # 4. SI-SNR in dB
        signal_power = (s_target * s_target).sum(dim=1) + self.eps          # [B]
        noise_power  = (e_noise  * e_noise ).sum(dim=1) + self.eps          # [B]
        sisnr        = 10 * torch.log10(signal_power / noise_power)         # [B]

        # Return mean over batch, negated (loss to minimise)
        return -sisnr.mean()

    def compute_metric(
        self,
        estimate: torch.Tensor,
        target:   torch.Tensor,
    ) -> float:
        """
        Returns SI-SNR in dB as a Python float (positive = good).
        Use this for logging/display.
        """
        with torch.no_grad():
            return -self.forward(estimate, target).item()


# =============================================================================
# SI-SDR METRIC  (for validation reporting)
# =============================================================================

def compute_si_sdr(
    estimate: torch.Tensor,   # [B, C, T]
    target:   torch.Tensor,   # [B, C, T]
    eps:      float = 1e-8,
) -> float:
    """
    Compute SI-SDR (dB) between estimate and target.
    Returns a Python float (mean over batch).

    SI-SDR and SI-SNR are equivalent formulas;
    we use SI-SDR as the standard evaluation metric.
    """
    with torch.no_grad():
        B, C, T  = estimate.shape
        est      = estimate.reshape(B, -1)
        tgt      = target.reshape(B, -1)

        est      = est - est.mean(dim=1, keepdim=True)
        tgt      = tgt - tgt.mean(dim=1, keepdim=True)

        dot      = (est * tgt).sum(dim=1, keepdim=True)
        tgt_pow  = (tgt * tgt).sum(dim=1, keepdim=True) + eps
        s_tgt    = dot / tgt_pow * tgt

        noise    = est - s_tgt
        si_sdr   = 10 * torch.log10(
            (s_tgt * s_tgt).sum(dim=1) / ((noise * noise).sum(dim=1) + eps)
        )
        return si_sdr.mean().item()


# =============================================================================
# COMBINED PHASE 1 LOSS
# =============================================================================

class Phase1Loss(nn.Module):
    """
    Combined loss for Phase 1 training.

    L_total = w_l1 * L1(pred_mag, target_mag)
            + w_sisnr * (-SI-SNR(pred_audio, target_audio))

    Why combine them?
    - L1 on magnitude: easy to optimise, guides early training
    - SI-SNR on audio: directly measures perceived quality

    Default weights: 70% L1, 30% SI-SNR
    (SI-SNR weight increases in Phase 4)
    """

    def __init__(self, w_l1: float = 0.7, w_sisnr: float = 0.3):
        super().__init__()
        self.w_l1    = w_l1
        self.w_sisnr = w_sisnr
        self.l1_loss    = L1MagnitudeLoss()
        self.sisnr_loss = SISNRLoss()

    def forward(
        self,
        pred_magnitude:   torch.Tensor,   # [B, C, F, T]
        target_magnitude: torch.Tensor,   # [B, C, F, T]
        pred_audio:       torch.Tensor,   # [B, C, T]
        target_audio:     torch.Tensor,   # [B, C, T]
    ) -> Dict[str, torch.Tensor]:
        """
        Returns a dict so we can log each component separately.
        {
            "loss"    : total weighted loss  (use this for .backward())
            "l1"      : L1 magnitude loss
            "sisnr"   : SI-SNR loss (negative dB)
        }
        """
        l1    = self.l1_loss(pred_magnitude, target_magnitude)
        sisnr = self.sisnr_loss(pred_audio, target_audio)
        total = self.w_l1 * l1 + self.w_sisnr * sisnr

        return {
            "loss":   total,
            "l1":     l1,
            "sisnr":  sisnr,
        }
'''

path = os.path.join(PROJECT_ROOT, "src", "training", "losses.py")
with open(path, 'w', encoding='utf-8') as f:
    f.write(losses_code)

print(f"✓ Written: {path}")
print(f"  Size: {os.path.getsize(path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\training\losses.py
  Size: 8,020 bytes


In [4]:
# Phase 1 - Cell 4: Write src/training/trainer.py

import os

trainer_code = '''"""
src/training/trainer.py
========================
Training loop for Phase 1 U-Net.

WHAT HAPPENS EACH TRAINING STEP:
  1. Load batch: (mixture_audio, target_audio)  from DataLoader
  2. Compute STFT: audio → magnitude + phase
  3. Forward pass: mixture_magnitude → predicted mask
  4. Apply mask: predicted_vocals_magnitude = mixture_magnitude * mask
  5. Compute loss: compare predicted vs target magnitudes + audio
  6. Backward pass: compute gradients
  7. Optimiser step: update model weights
  8. Repeat!

WHAT IS AN EPOCH?
  One full pass through the training dataset.
  e.g., 1000 songs × 10 chunks each = 10,000 items
        with batch_size=4: 2,500 steps per epoch

WHAT IS A CHECKPOINT?
  A saved copy of the model weights at a point in time.
  We save the best checkpoint (lowest validation loss).
  If training crashes, we can resume from the last checkpoint.
"""

import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from typing import Dict, Optional, Tuple
from dataclasses import dataclass, field


@dataclass
class TrainerConfig:
    """All training hyperparameters."""

    # Optimiser
    learning_rate: float = 1e-3      # How big each weight update is
    weight_decay:  float = 1e-4      # L2 regularisation (prevents overfitting)

    # Scheduler: reduce LR when validation loss stops improving
    lr_patience:  int   = 5          # Wait this many epochs before reducing LR
    lr_factor:    float = 0.5        # Multiply LR by this when reducing
    min_lr:       float = 1e-6       # Never go below this LR

    # Training duration
    max_epochs:   int   = 100
    grad_clip:    float = 5.0        # Clip gradients to prevent explosion

    # Saving
    checkpoint_dir: str = "checkpoints"
    save_every:     int = 5          # Save checkpoint every N epochs

    # Logging
    log_every:    int   = 10         # Print loss every N batches

    # Early stopping
    patience:     int   = 20         # Stop if no improvement for N epochs


# =============================================================================
# METRIC TRACKER
# =============================================================================

class MetricTracker:
    """
    Tracks running average of metrics during an epoch.

    Usage:
        tracker = MetricTracker()
        for batch in loader:
            loss = compute_loss(batch)
            tracker.update({"loss": loss.item(), "si_sdr": si_sdr})
        print(tracker.averages())
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self._sums   = {}
        self._counts = {}

    def update(self, metrics: Dict[str, float]) -> None:
        for k, v in metrics.items():
            if k not in self._sums:
                self._sums[k]   = 0.0
                self._counts[k] = 0
            self._sums[k]   += float(v)
            self._counts[k] += 1

    def averages(self) -> Dict[str, float]:
        return {
            k: self._sums[k] / self._counts[k]
            for k in self._sums
            if self._counts[k] > 0
        }

    def average(self, key: str) -> float:
        if key not in self._sums or self._counts[key] == 0:
            return float("nan")
        return self._sums[key] / self._counts[key]


# =============================================================================
# TRAINER
# =============================================================================

class Trainer:
    """
    Manages the full training loop for Phase 1.

    Usage:
        trainer = Trainer(model, processor, loss_fn, config, device)
        trainer.fit(train_loader, val_loader, epochs=50)
    """

    def __init__(
        self,
        model:      nn.Module,
        processor,                  # STFTProcessor
        loss_fn:    nn.Module,
        config:     TrainerConfig,
        device:     torch.device,
    ):
        self.model     = model.to(device)
        self.processor = processor.to(device)
        self.loss_fn   = loss_fn
        self.config    = config
        self.device    = device

        # Optimiser: Adam is the standard choice
        self.optimiser = Adam(
            model.parameters(),
            lr           = config.learning_rate,
            weight_decay = config.weight_decay,
        )

        # Scheduler: reduce LR when stuck
        self.scheduler = ReduceLROnPlateau(
            self.optimiser,
            mode      = "min",           # reduce when val_loss stops decreasing
            patience  = config.lr_patience,
            factor    = config.lr_factor,
            min_lr    = config.min_lr,
        )

        # State tracking
        self.best_val_loss    = float("inf")
        self.epochs_no_improve = 0
        self.history: Dict[str, list] = {
            "train_loss": [], "val_loss": [],
            "train_si_sdr": [], "val_si_sdr": [],
            "lr": [],
        }

        # Make checkpoint dir
        os.makedirs(config.checkpoint_dir, exist_ok=True)

    # ------------------------------------------------------------------
    def fit(
        self,
        train_loader: DataLoader,
        val_loader:   DataLoader,
        epochs:       Optional[int] = None,
    ) -> Dict[str, list]:
        """
        Run the full training loop.

        Args:
            train_loader : training DataLoader
            val_loader   : validation DataLoader
            epochs       : override max_epochs if provided

        Returns:
            history dict with loss curves
        """
        max_epochs = epochs or self.config.max_epochs

        print("="*60)
        print("PHASE 1 TRAINING START")
        print("="*60)
        print(f"  Model params : {self.model.count_parameters():,}")
        print(f"  Train batches: {len(train_loader)}")
        print(f"  Val batches  : {len(val_loader)}")
        print(f"  Max epochs   : {max_epochs}")
        print(f"  Learning rate: {self.config.learning_rate}")
        print(f"  Device       : {self.device}")
        print("="*60)

        for epoch in range(1, max_epochs + 1):
            t0 = time.time()

            # ── Train ───────────────────────────────────────────────
            train_metrics = self._train_epoch(train_loader, epoch)

            # ── Validate ────────────────────────────────────────────
            val_metrics = self._val_epoch(val_loader)

            elapsed = time.time() - t0

            # ── Scheduler step ──────────────────────────────────────
            self.scheduler.step(val_metrics["loss"])
            current_lr = self.optimiser.param_groups[0]["lr"]

            # ── Log ─────────────────────────────────────────────────
            self.history["train_loss"].append(train_metrics["loss"])
            self.history["val_loss"].append(val_metrics["loss"])
            self.history["train_si_sdr"].append(train_metrics.get("si_sdr", 0))
            self.history["val_si_sdr"].append(val_metrics.get("si_sdr", 0))
            self.history["lr"].append(current_lr)

            print(
                f"Epoch {epoch:03d}/{max_epochs} "
                f"| train_loss={train_metrics['loss']:.4f} "
                f"| val_loss={val_metrics['loss']:.4f} "
                f"| val_SI-SDR={val_metrics.get('si_sdr', 0):+.2f}dB "
                f"| lr={current_lr:.1e} "
                f"| {elapsed:.1f}s"
            )

            # ── Save best checkpoint ─────────────────────────────────
            if val_metrics["loss"] < self.best_val_loss:
                self.best_val_loss     = val_metrics["loss"]
                self.epochs_no_improve = 0
                self._save_checkpoint(epoch, val_metrics, is_best=True)
                print(f"  ★ New best! val_loss={self.best_val_loss:.4f}")
            else:
                self.epochs_no_improve += 1

            # ── Periodic checkpoint ──────────────────────────────────
            if epoch % self.config.save_every == 0:
                self._save_checkpoint(epoch, val_metrics, is_best=False)

            # ── Early stopping ───────────────────────────────────────
            if self.epochs_no_improve >= self.config.patience:
                print(f"\\nEarly stopping at epoch {epoch} "
                      f"(no improvement for {self.config.patience} epochs)")
                break

        print("\\nTraining complete!")
        print(f"Best val_loss : {self.best_val_loss:.4f}")
        return self.history

    # ------------------------------------------------------------------
    def _train_epoch(
        self,
        loader: DataLoader,
        epoch:  int,
    ) -> Dict[str, float]:
        """Run one training epoch."""
        self.model.train()
        tracker = MetricTracker()

        for batch_idx, (mix_audio, tgt_audio) in enumerate(loader):
            # Move to device
            mix_audio = mix_audio.to(self.device)   # [B, C, T]
            tgt_audio = tgt_audio.to(self.device)   # [B, C, T]

            # ── STFT ──────────────────────────────────────────────
            with torch.no_grad():   # STFT has no learnable params
                mix_mag, mix_phase, orig_len = self.processor(mix_audio)
                tgt_mag, _,         _        = self.processor(tgt_audio)
            # mix_mag: [B, C, F, T]

            # ── Forward pass ──────────────────────────────────────
            # Process each channel: [B, 1, F, T] → mask → [B, 1, F, T]
            B, C, F, T = mix_mag.shape
            pred_masks = []
            for ch in range(C):
                ch_mag  = mix_mag[:, ch:ch+1, :, :]   # [B, 1, F, T]
                ch_mask = self.model(ch_mag)            # [B, 1, F, T]
                pred_masks.append(ch_mask)

            pred_mask = torch.cat(pred_masks, dim=1)   # [B, C, F, T]

            # Apply mask to get predicted magnitude
            pred_mag  = mix_mag * pred_mask             # [B, C, F, T]

            # Reconstruct audio for SI-SNR loss
            pred_stft  = self.processor.magnitude_phase_to_complex(pred_mag, mix_phase)
            pred_audio = self.processor.istft(pred_stft, length=orig_len)   # [B, C, T]

            # ── Loss ──────────────────────────────────────────────
            loss_dict = self.loss_fn(pred_mag, tgt_mag, pred_audio, tgt_audio)
            loss      = loss_dict["loss"]

            # ── Backward ──────────────────────────────────────────
            self.optimiser.zero_grad()
            loss.backward()

            # Gradient clipping: prevents exploding gradients
            nn.utils.clip_grad_norm_(
                self.model.parameters(), self.config.grad_clip
            )

            self.optimiser.step()

            # ── Metrics ───────────────────────────────────────────
            from src.training.losses import compute_si_sdr
            si_sdr = compute_si_sdr(pred_audio.detach(), tgt_audio)

            tracker.update({
                "loss":   loss.item(),
                "l1":     loss_dict["l1"].item(),
                "sisnr":  loss_dict["sisnr"].item(),
                "si_sdr": si_sdr,
            })

            # ── Log ───────────────────────────────────────────────
            if batch_idx % self.config.log_every == 0:
                avgs = tracker.averages()
                print(
                    f"  [{epoch}] batch {batch_idx:04d}/{len(loader)} "
                    f"loss={avgs[\'loss\']:.4f} "
                    f"SI-SDR={avgs[\'si_sdr\']:+.2f}dB",
                    end="\\r"
                )

        print()  # newline after \\r
        return tracker.averages()

    # ------------------------------------------------------------------
    @torch.no_grad()
    def _val_epoch(self, loader: DataLoader) -> Dict[str, float]:
        """Run one validation epoch (no gradients)."""
        self.model.eval()
        tracker = MetricTracker()

        for mix_audio, tgt_audio in loader:
            mix_audio = mix_audio.to(self.device)
            tgt_audio = tgt_audio.to(self.device)

            mix_mag, mix_phase, orig_len = self.processor(mix_audio)
            tgt_mag, _,         _        = self.processor(tgt_audio)

            B, C, F, T = mix_mag.shape
            pred_masks = []
            for ch in range(C):
                ch_mask = self.model(mix_mag[:, ch:ch+1, :, :])
                pred_masks.append(ch_mask)

            pred_mask  = torch.cat(pred_masks, dim=1)
            pred_mag   = mix_mag * pred_mask

            pred_stft  = self.processor.magnitude_phase_to_complex(pred_mag, mix_phase)
            pred_audio = self.processor.istft(pred_stft, length=orig_len)

            loss_dict  = self.loss_fn(pred_mag, tgt_mag, pred_audio, tgt_audio)

            from src.training.losses import compute_si_sdr
            si_sdr = compute_si_sdr(pred_audio, tgt_audio)

            tracker.update({
                "loss":   loss_dict["loss"].item(),
                "l1":     loss_dict["l1"].item(),
                "sisnr":  loss_dict["sisnr"].item(),
                "si_sdr": si_sdr,
            })

        return tracker.averages()

    # ------------------------------------------------------------------
    def _save_checkpoint(
        self,
        epoch:      int,
        metrics:    Dict[str, float],
        is_best:    bool,
    ) -> None:
        """Save model weights + training state."""
        state = {
            "epoch":         epoch,
            "model_state":   self.model.state_dict(),
            "optim_state":   self.optimiser.state_dict(),
            "metrics":       metrics,
            "best_val_loss": self.best_val_loss,
            "history":       self.history,
        }

        if is_best:
            path = os.path.join(self.config.checkpoint_dir, "best_model.pt")
        else:
            path = os.path.join(
                self.config.checkpoint_dir, f"epoch_{epoch:03d}.pt"
            )

        torch.save(state, path)

    # ------------------------------------------------------------------
    def load_checkpoint(self, path: str) -> int:
        """
        Load a checkpoint to resume training.

        Args:
            path : path to .pt file

        Returns:
            epoch number where training left off
        """
        state = torch.load(path, map_location=self.device)

        self.model.load_state_dict(state["model_state"])
        self.optimiser.load_state_dict(state["optim_state"])
        self.best_val_loss = state["best_val_loss"]
        self.history       = state.get("history", self.history)

        epoch = state["epoch"]
        print(f"✓ Loaded checkpoint from epoch {epoch}")
        print(f"  Best val_loss so far: {self.best_val_loss:.4f}")
        return epoch
'''

path = os.path.join(PROJECT_ROOT, "src", "training", "trainer.py")
with open(path, 'w', encoding='utf-8') as f:
    f.write(trainer_code)

print(f"✓ Written: {path}")
print(f"  Size: {os.path.getsize(path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\training\trainer.py
  Size: 16,292 bytes


In [5]:
# Phase 1 - Cell 5: Test the U-Net model BEFORE any training
# Always do this - catches bugs before wasting GPU time

import os, sys
for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

import torch
import torch.nn as nn
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp  import STFTConfig, STFTProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# ── Build model ──────────────────────────────────────────────
cfg   = UNetConfig(n_freq_bins=1025, base_channels=32, depth=4, dropout=0.1)
model = MagnitudeUNet(cfg).to(device)

print("U-Net Architecture:")
print(f"  base_channels : {cfg.base_channels}")
print(f"  depth         : {cfg.depth}")
print(f"  channel sizes : {[cfg.base_channels * 2**i for i in range(cfg.depth)]}")
print(f"  parameters    : {model.count_parameters():,}")
print()

# ── Test forward pass ─────────────────────────────────────────
# Fake a realistic batch
batch_size   = 2
n_freq_bins  = 1025     # n_fft//2 + 1
time_frames  = 345      # ~4s at 44.1kHz with hop=512

print(f"Test input: [B={batch_size}, C=1, F={n_freq_bins}, T={time_frames}]")

x    = torch.rand(batch_size, 1, n_freq_bins, time_frames).to(device)
print(f"  Input  range: [{x.min():.3f}, {x.max():.3f}]")

with torch.no_grad():
    mask = model(x)

print(f"  Output shape: {list(mask.shape)}")
print(f"  Output range: [{mask.min():.4f}, {mask.max():.4f}]")
print(f"  Output is in [0,1]: {(mask >= 0).all() and (mask <= 1).all()}")

assert mask.shape == x.shape,         f"Shape mismatch! {mask.shape} vs {x.shape}"
assert (mask >= 0).all(),             "Mask has negative values!"
assert (mask <= 1).all(),             "Mask exceeds 1.0!"
assert not torch.isnan(mask).any(),   "NaN in mask output!"

print("\n✓ Forward pass PASSED")

# ── Test backward pass (gradient flow) ───────────────────────
print("\nTesting gradient flow...")
model.train()
x2   = torch.rand(2, 1, 1025, 345).to(device)
tgt  = torch.rand(2, 1, 1025, 345).to(device)
mask2 = model(x2)
pred  = x2 * mask2
loss  = torch.nn.functional.l1_loss(pred, tgt)
loss.backward()

# Check that all parameters received gradients
no_grad = [
    name for name, p in model.named_parameters()
    if p.requires_grad and p.grad is None
]
if no_grad:
    print(f"  ⚠ Parameters with no gradient: {no_grad}")
else:
    print(f"  ✓ All {model.count_parameters():,} parameters have gradients")
    print(f"  ✓ Loss = {loss.item():.4f}")

print("\n✓ Backward pass PASSED")

# ── Memory estimate ───────────────────────────────────────────
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    before = torch.cuda.memory_allocated()
    _ = model(torch.rand(4, 1, 1025, 345).to(device))
    after  = torch.cuda.memory_allocated()
    print(f"\nGPU memory for batch of 4: {(after-before)/1e6:.1f} MB")

Device: cuda

U-Net Architecture:
  base_channels : 32
  depth         : 4
  channel sizes : [32, 64, 128, 256]
  parameters    : 3,349,697

Test input: [B=2, C=1, F=1025, T=345]
  Input  range: [0.000, 1.000]
  Output shape: [2, 1, 1025, 345]
  Output range: [0.1289, 0.8797]
  Output is in [0,1]: True

✓ Forward pass PASSED

Testing gradient flow...
  ✓ All 3,349,697 parameters have gradients
  ✓ Loss = 0.3386

✓ Backward pass PASSED

GPU memory for batch of 4: 8847.6 MB


In [7]:
# Phase 1 - Cell 6 (FIXED): Test loss functions with known inputs

import sys, os
for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
from src.training.losses import (
    L1MagnitudeLoss, SISNRLoss, Phase1Loss, compute_si_sdr
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Testing Loss Functions")
print("="*55)

# ── WHY THE ERROR HAPPENED ────────────────────────────────────
# torch.rand() and torch.randn() create tensors with requires_grad=False
# .backward() needs at least ONE tensor in the graph to have requires_grad=True
# Solution: mark the PREDICTION tensors as requiring gradients
#           (targets never need gradients - we don't optimise them)

# ── [1] L1 Magnitude Loss ─────────────────────────────────────
print("\n[1] L1 Magnitude Loss")
l1_fn = L1MagnitudeLoss()

# Perfect prediction → loss = 0
perfect = torch.rand(2, 2, 1025, 100)
loss_perfect = l1_fn(perfect, perfect)
print(f"  Perfect prediction loss : {loss_perfect.item():.6f}  (should be ~0)")
assert loss_perfect.item() < 1e-5

# Random prediction
# requires_grad=True on pred → PyTorch builds the computation graph
pred = torch.rand(2, 2, 1025, 100, requires_grad=True)
tgt  = torch.rand(2, 2, 1025, 100)                      # target never needs grad
loss_rand = l1_fn(pred, tgt)
print(f"  Random prediction loss  : {loss_rand.item():.4f}  (expect ~0.33)")

# Test backward
loss_rand.backward()
print(f"  pred.grad norm          : {pred.grad.norm().item():.4f}  (should be > 0)")
assert pred.grad is not None, "No gradient on pred!"
assert pred.grad.norm().item() > 0,  "Gradient is zero!"
print("  ✓ L1 loss + backward works")

# ── [2] SI-SNR Loss ───────────────────────────────────────────
print("\n[2] SI-SNR Loss")
sisnr_fn = SISNRLoss()

# Perfect reconstruction
audio_ref  = torch.randn(2, 2, 44100)
loss_ideal = sisnr_fn(audio_ref, audio_ref)
sisnr_db   = -loss_ideal.item()
print(f"  Perfect SI-SNR : {sisnr_db:.1f} dB  (should be >> 30 dB)")
assert sisnr_db > 30.0, f"Expected > 30 dB, got {sisnr_db:.1f}"

# Good vs bad separation (metric only - no backward needed here)
clean     = torch.randn(2, 2, 44100)
good_est  = clean + 0.1 * torch.randn_like(clean)   # mostly signal
bad_est   = 0.1 * clean + torch.randn_like(clean)   # mostly noise

si_sdr_good = compute_si_sdr(good_est, clean)
si_sdr_bad  = compute_si_sdr(bad_est,  clean)
print(f"  Good separation SI-SDR  : {si_sdr_good:+.1f} dB  (expect ~+20 dB)")
print(f"  Bad  separation SI-SDR  : {si_sdr_bad:+.1f} dB  (expect negative)")
assert si_sdr_good > si_sdr_bad, "Good estimate must have higher SI-SDR!"

# Test backward on SI-SNR
# requires_grad=True so the graph is built
est_grad = torch.randn(2, 2, 44100, requires_grad=True)
tgt_fix  = torch.randn(2, 2, 44100)                    # no grad on target
loss_sisnr = sisnr_fn(est_grad, tgt_fix)
loss_sisnr.backward()
print(f"  est_grad.grad norm      : {est_grad.grad.norm().item():.4f}  (should be > 0)")
assert est_grad.grad is not None
assert est_grad.grad.norm().item() > 0
print("  ✓ SI-SNR loss + backward works")

# ── [3] Phase1Loss (combined) ─────────────────────────────────
print("\n[3] Phase1Loss (combined)")
p1_loss = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# Simulate ONE forward pass through a tiny fake model
# so the predictions are connected to a computation graph
tiny_model = nn.Sequential(
    nn.Conv2d(1, 1, kernel_size=1),
    nn.Sigmoid(),
)

# --- Magnitude inputs ---
mix_mag_np = torch.rand(2, 2, 1025, 50)   # [B, C, F, T]  no grad needed
tgt_mag    = torch.rand(2, 2, 1025, 50)   # target - no grad

# Predict mask per channel through tiny_model (has grad_fn)
ch0_mask = tiny_model(mix_mag_np[:, 0:1, :, :])   # [B,1,F,T]
ch1_mask = tiny_model(mix_mag_np[:, 1:2, :, :])
pred_mask = torch.cat([ch0_mask, ch1_mask], dim=1) # [B,2,F,T]
pred_mag  = mix_mag_np * pred_mask                  # [B,2,F,T]  has grad_fn ✓

# --- Audio inputs ---
# Fake "iSTFT" result: just a linear layer so gradient flows
audio_proj = nn.Linear(50, 44100, bias=False)
# flatten [B,C,F,T] → [B, C*F*T] → project → [B, C*T] → reshape
B, C, F, T = pred_mag.shape
pred_flat  = pred_mag.reshape(B, -1)                # [B, C*F*T]

# Simple fake audio: sum over freq bins (keeps grad_fn)
pred_audio = pred_mag.mean(dim=2)                   # [B, C, T]  has grad_fn ✓
tgt_audio  = torch.randn(B, C, T)                  # target - no grad

# --- Forward through loss ---
result = p1_loss(pred_mag, tgt_mag, pred_audio, tgt_audio)

print(f"  Total loss : {result['loss'].item():.4f}")
print(f"  L1 loss    : {result['l1'].item():.4f}")
print(f"  SI-SNR loss: {result['sisnr'].item():.4f}")
print(f"  has grad_fn: {result['loss'].grad_fn is not None}  ← must be True")

assert result['loss'].grad_fn is not None, (
    "Loss has no grad_fn! "
    "Make sure predictions come from a model, not plain torch.rand()"
)

# Backward pass
result['loss'].backward()
print("  ✓ Gradients computed successfully")

# Check model gradients
for name, param in tiny_model.named_parameters():
    if param.grad is not None:
        print(f"  ✓ tiny_model.{name}.grad norm = {param.grad.norm().item():.4f}")

# ── [4] Verify in a real mini training step ───────────────────
print("\n[4] End-to-end mini training step (closest to real training)")

# Import the actual U-Net
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp   import STFTConfig, STFTProcessor

stft_cfg  = STFTConfig()
processor = STFTProcessor(stft_cfg).to(device)

unet_cfg  = UNetConfig(n_freq_bins=1025, base_channels=16, depth=2)  # tiny for speed
model     = MagnitudeUNet(unet_cfg).to(device)
optim     = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn2  = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# Fake audio batch: [B=2, C=2, T=4*44100]
torch.manual_seed(42)
fake_mix = torch.randn(2, 2, 4 * 44100).to(device)
fake_tgt = torch.randn(2, 2, 4 * 44100).to(device)

# STFT
mix_mag, mix_phase, orig_len = processor(fake_mix)   # [2, 2, 1025, T]
tgt_mag, _,         _        = processor(fake_tgt)

# Forward: per-channel masking
B, C, F, Tf = mix_mag.shape
masks     = [model(mix_mag[:, c:c+1]) for c in range(C)]
pred_mask = torch.cat(masks, dim=1)                  # [B, C, F, T]
pred_mag2 = mix_mag * pred_mask                      # [B, C, F, T]

# iSTFT to audio
pred_stft  = processor.magnitude_phase_to_complex(pred_mag2, mix_phase)
pred_audio2 = processor.istft(pred_stft, length=orig_len)   # [B, C, T]

print(f"  pred_mag  grad_fn : {pred_mag2.grad_fn.__class__.__name__}")
print(f"  pred_audio grad_fn: {pred_audio2.grad_fn.__class__.__name__}")

# Loss + backward
losses2 = loss_fn2(pred_mag2, tgt_mag, pred_audio2, fake_tgt)
optim.zero_grad()
losses2['loss'].backward()
optim.step()

# Check model params got gradients
grads_ok = all(
    p.grad is not None
    for p in model.parameters() if p.requires_grad
)
print(f"  All model params have grad: {grads_ok}")
print(f"  loss  = {losses2['loss'].item():.4f}")
print(f"  l1    = {losses2['l1'].item():.4f}")
print(f"  sisnr = {losses2['sisnr'].item():.4f}")
assert grads_ok, "Some parameters have no gradient!"

print("\n✓ Mini training step PASSED")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*55)
print("ALL LOSS TESTS PASSED ✓")
print("="*55)
print("""
Key lesson:
  targets  → requires_grad=False  (we never optimise these)
  predictions → requires_grad=True  (always comes from model forward pass)

In real training (Cell 8) this is automatic because:
  model(input) always produces tensors with grad_fn attached.
""")

Testing Loss Functions

[1] L1 Magnitude Loss
  Perfect prediction loss : 0.000000  (should be ~0)
  Random prediction loss  : 0.3340  (expect ~0.33)
  pred.grad norm          : 0.0016  (should be > 0)
  ✓ L1 loss + backward works

[2] SI-SNR Loss
  Perfect SI-SNR : 129.4 dB  (should be >> 30 dB)
  Good separation SI-SDR  : +20.0 dB  (expect ~+20 dB)
  Bad  separation SI-SDR  : -20.0 dB  (expect negative)
  est_grad.grad norm      : 7.2695  (should be > 0)
  ✓ SI-SNR loss + backward works

[3] Phase1Loss (combined)
  Total loss : 7.6345
  L1 loss    : 0.3485
  SI-SNR loss: 24.6350
  has grad_fn: True  ← must be True
  ✓ Gradients computed successfully
  ✓ tiny_model.0.weight.grad norm = 1.5150
  ✓ tiny_model.0.bias.grad norm = 0.2115

[4] End-to-end mini training step (closest to real training)
  pred_mag  grad_fn : MulBackward0
  pred_audio grad_fn: ViewBackward0
  All model params have grad: True
  loss  = 33.5237
  l1    = 15.5932
  sisnr = 75.3616

✓ Mini training step PASSED

ALL 

In [8]:
# Phase 1 - Cell 7: Smoke test - train for 2 batches to make sure nothing crashes

import os, sys, torch
for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
MUSDB_PATH   = "D:/dataset"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.unet           import MagnitudeUNet, UNetConfig
from src.audio.dsp             import STFTConfig, STFTProcessor
from src.training.losses       import Phase1Loss
from src.data.musdb_dataset    import DataConfig, MUSDB18Dataset
from src.audio.dsp             import AudioAugmenter
from torch.utils.data          import DataLoader
from torch.optim               import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print("SMOKE TEST: 2-batch training check")
print("="*50)

# ── Build everything ─────────────────────────────────────────
stft_cfg  = STFTConfig()
processor = STFTProcessor(stft_cfg).to(device)

unet_cfg  = UNetConfig(n_freq_bins=1025, base_channels=32, depth=4)
model     = MagnitudeUNet(unet_cfg).to(device)

loss_fn   = Phase1Loss(w_l1=0.7, w_sisnr=0.3)
optimiser = Adam(model.parameters(), lr=1e-3)

# ── Dataset (small batch for smoke test) ─────────────────────
aug      = AudioAugmenter(gain_range=(0.8, 1.2), swap_prob=0.5, seed=42)
dcfg     = DataConfig(dataset_path=MUSDB_PATH, batch_size=2, num_workers=0)
ds       = MUSDB18Dataset(dcfg, "train", "vocals", aug)
loader   = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)

print(f"Model params : {model.count_parameters():,}")
print(f"Dataset size : {len(ds)}")
print()

# ── Run 2 batches ────────────────────────────────────────────
model.train()
for step, (mix_audio, tgt_audio) in enumerate(loader):
    if step >= 2:
        break

    mix_audio = mix_audio.to(device)   # [B, C, T]
    tgt_audio = tgt_audio.to(device)

    print(f"Step {step+1}:")
    print(f"  mix_audio  : {list(mix_audio.shape)}")

    # STFT
    mix_mag, mix_phase, orig_len = processor(mix_audio)
    tgt_mag, _,         _        = processor(tgt_audio)
    print(f"  mix_mag    : {list(mix_mag.shape)}")

    # Forward: per-channel masking
    B, C, F, T = mix_mag.shape
    masks = [model(mix_mag[:, c:c+1]) for c in range(C)]
    mask  = torch.cat(masks, dim=1)             # [B, C, F, T]
    pred_mag = mix_mag * mask

    # Audio reconstruction
    pred_stft  = processor.magnitude_phase_to_complex(pred_mag, mix_phase)
    pred_audio = processor.istft(pred_stft, length=orig_len)

    # Loss
    losses = loss_fn(pred_mag, tgt_mag, pred_audio, tgt_audio)
    loss   = losses["loss"]

    # Backward
    optimiser.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimiser.step()

    from src.training.losses import compute_si_sdr
    si_sdr = compute_si_sdr(pred_audio.detach(), tgt_audio)

    print(f"  loss       : {loss.item():.4f}")
    print(f"  l1         : {losses['l1'].item():.4f}")
    print(f"  SI-SDR     : {si_sdr:+.2f} dB")
    print(f"  mask range : [{mask.min():.3f}, {mask.max():.3f}]")
    assert not torch.isnan(loss), "NaN loss detected!"
    print(f"  ✓ No NaN\n")

print("✓ SMOKE TEST PASSED - ready for full training!")

SMOKE TEST: 2-batch training check

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)
Model params : 3,349,697
Dataset size : 1000

Step 1:
  mix_audio  : [2, 2, 176400]
  mix_mag    : [2, 2, 1025, 345]
  loss       : 17.9952
  l1         : 0.4084
  SI-SDR     : -inf dB
  mask range : [0.000, 1.000]
  ✓ No NaN

Step 2:
  mix_audio  : [2, 2, 176400]
  mix_mag    : [2, 2, 1025, 345]
  loss       : 10.7736
  l1         : 0.1739
  SI-SDR     : -35.51 dB
  mask range : [0.000, 0.999]
  ✓ No NaN

✓ SMOKE TEST PASSED - ready for full training!


In [3]:
# Fix Cell 1: GPU memory audit
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Name       : {props.name}")
    print(f"Total VRAM     : {props.total_memory / 1e9:.2f} GB")
    print(f"Currently used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Reserved       : {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    free = props.total_memory - torch.cuda.memory_allocated()
    print(f"Free (approx)  : {free / 1e9:.2f} GB")
    print()
    
    # Recommend settings based on VRAM
    vram_gb = props.total_memory / 1e9
    print("RECOMMENDED SETTINGS FOR YOUR GPU:")
    print("-"*45)
    if vram_gb <= 4:
        print("  batch_size     = 1")
        print("  base_channels  = 16")
        print("  depth          = 3")
        print("  chunk_duration = 2.0")
    elif vram_gb <= 6:
        print("  batch_size     = 2  (or 4 with gradient checkpointing)")
        print("  base_channels  = 24")
        print("  depth          = 4")
        print("  chunk_duration = 3.0")
        print("  downsample BOTH freq AND time axes  ← key fix!")
    elif vram_gb <= 8:
        print("  batch_size     = 4")
        print("  base_channels  = 32")
        print("  depth          = 4")
        print("  chunk_duration = 4.0")
    else:
        print("  batch_size     = 8")
        print("  base_channels  = 32")
        print("  depth          = 4")
        print("  chunk_duration = 4.0")

GPU Name       : NVIDIA GeForce RTX 3050 6GB Laptop GPU
Total VRAM     : 6.44 GB
Currently used : 0.00 GB
Reserved       : 0.00 GB
Free (approx)  : 6.44 GB

RECOMMENDED SETTINGS FOR YOUR GPU:
---------------------------------------------
  batch_size     = 4
  base_channels  = 32
  depth          = 4
  chunk_duration = 4.0


In [4]:
# Fix Cell 2: Memory-efficient U-Net for 6GB GPU
# Key changes:
#   1. Pool on BOTH freq AND time (2x2 instead of 1x2)
#   2. Added gradient checkpointing option
#   3. Configurable for any GPU size

import os

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

unet_code = '''"""
src/models/unet.py
==================
Memory-Efficient Magnitude U-Net for source separation.

KEY CHANGE vs original:
  Original : MaxPool2d(1, 2) → only downsamples TIME axis
             → freq axis stays at 1025 throughout → OOM on 6GB GPU

  Fixed    : MaxPool2d(2, 2) → downsamples BOTH freq AND time
             → feature maps shrink 4x per layer → fits in 6GB

Memory comparison (batch=4, depth=4, base=24):
  Original : ~12 GB (OOM on 6GB)
  Fixed    : ~2.5 GB (fits comfortably)

Architecture for 6GB GPU:
  Input  : [B, 1, F,    T   ]  e.g. [4, 1, 1025, 345]
  Enc 1  : [B, 24, F//2, T//2]     [4, 24, 512,  172]
  Enc 2  : [B, 48, F//4, T//4]     [4, 48, 256,  86 ]
  Enc 3  : [B, 96, F//8, T//8]     [4, 96, 128,  43 ]
  Enc 4  : [B, 192,F//16,T//16]    [4, 192,64,   21 ]
  Bottle : [B, 192,F//16,T//16]    [4, 192,64,   21 ]
  Dec 4  : [B, 96, F//8, T//8]     (upsample + skip)
  Dec 3  : [B, 48, F//4, T//4]
  Dec 2  : [B, 24, F//2, T//2]
  Dec 1  : [B, 24, F,    T   ]
  Out    : [B, 1,  F,    T   ]  mask in [0,1]
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import math


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class UNetConfig:
    """
    U-Net hyperparameters.

    pool_freq : whether to downsample frequency axis (True = memory efficient)
    use_grad_checkpoint : trades compute for memory (good for small GPUs)
    """
    n_freq_bins:          int   = 1025
    base_channels:        int   = 24       # 24→48→96→192 for 6GB GPU
    depth:                int   = 4
    dropout:              float = 0.1
    pool_freq:            bool  = True     # ← KEY: downsample freq axis too
    use_grad_checkpoint:  bool  = False    # extra memory saving if needed

    @property
    def channel_list(self) -> List[int]:
        """Channel sizes at each depth level."""
        return [self.base_channels * (2 ** i) for i in range(self.depth)]

    # Preset configs for different GPU sizes
    @classmethod
    def for_4gb_gpu(cls) -> "UNetConfig":
        return cls(base_channels=16, depth=3, dropout=0.1,
                   pool_freq=True, use_grad_checkpoint=True)

    @classmethod
    def for_6gb_gpu(cls) -> "UNetConfig":
        return cls(base_channels=24, depth=4, dropout=0.1,
                   pool_freq=True, use_grad_checkpoint=False)

    @classmethod
    def for_8gb_gpu(cls) -> "UNetConfig":
        return cls(base_channels=32, depth=4, dropout=0.1,
                   pool_freq=True, use_grad_checkpoint=False)

    @classmethod
    def for_12gb_gpu(cls) -> "UNetConfig":
        return cls(base_channels=32, depth=4, dropout=0.1,
                   pool_freq=False, use_grad_checkpoint=False)


# =============================================================================
# BUILDING BLOCKS
# =============================================================================

class ConvBlock(nn.Module):
    """
    Two Conv2d layers with BatchNorm + LeakyReLU.
    Shape: [B, C_in, F, T] → [B, C_out, F, T]  (same spatial size)
    """

    def __init__(
        self,
        in_channels:  int,
        out_channels: int,
        dropout:      float = 0.1,
    ):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels,  out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout),

            nn.Conv2d(out_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C_in, F, T] → [B, C_out, F, T]
        return self.block(x)


class EncoderBlock(nn.Module):
    """
    ConvBlock + MaxPool2d downsampling.

    pool_size controls what gets downsampled:
      (2, 2) → halve both freq AND time  ← memory efficient
      (1, 2) → halve time only           ← original (OOM on small GPU)

    Returns:
        downsampled : [B, C_out, F//p, T//p]   (passed to next encoder)
        skip        : [B, C_out, F,    T   ]   (saved for decoder)
    """

    def __init__(
        self,
        in_channels:  int,
        out_channels: int,
        dropout:      float = 0.1,
        pool_freq:    bool  = True,
    ):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels, dropout)
        pool_size = (2, 2) if pool_freq else (1, 2)
        self.pool = nn.MaxPool2d(kernel_size=pool_size)

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        skip        = self.conv(x)          # [B, C_out, F, T]
        downsampled = self.pool(skip)       # [B, C_out, F//p, T//p]
        return downsampled, skip


class DecoderBlock(nn.Module):
    """
    Upsample → concat skip → ConvBlock.

    Input  : [B, C_in,   F//p, T//p]   from previous decoder
    Skip   : [B, C_skip, F,    T   ]   from matching encoder
    Output : [B, C_out,  F,    T   ]
    """

    def __init__(
        self,
        in_channels:   int,
        skip_channels: int,
        out_channels:  int,
        dropout:       float = 0.1,
    ):
        super().__init__()
        # After concat: in_channels + skip_channels → out_channels
        self.conv = ConvBlock(in_channels + skip_channels, out_channels, dropout)

    def forward(
        self,
        x:    torch.Tensor,   # [B, C_in,   F//p, T//p]
        skip: torch.Tensor,   # [B, C_skip, F,    T   ]
    ) -> torch.Tensor:
        # 1. Upsample to match skip spatial size
        x = F.interpolate(x, size=skip.shape[2:],
                          mode="bilinear", align_corners=False)
        # x: [B, C_in, F, T]

        # 2. Concat along channel dim
        x = torch.cat([x, skip], dim=1)
        # x: [B, C_in + C_skip, F, T]

        # 3. Fuse with ConvBlock
        return self.conv(x)
        # output: [B, C_out, F, T]


# =============================================================================
# MAIN U-NET
# =============================================================================

class MagnitudeUNet(nn.Module):
    """
    Memory-efficient U-Net for magnitude spectrogram masking.

    Input  : mixture magnitude [B, 1, F, T]
    Output : soft mask         [B, 1, F, T]  in [0, 1]

    Apply mask:
        separated_mag = mixture_magnitude * mask
    """

    def __init__(self, config: UNetConfig):
        super().__init__()
        self.config = config
        ch = config.channel_list   # e.g. [24, 48, 96, 192]

        # ── Encoder ──────────────────────────────────────────────────
        self.encoders = nn.ModuleList()
        # First encoder: 1 → ch[0]
        self.encoders.append(
            EncoderBlock(1, ch[0], config.dropout, config.pool_freq)
        )
        # Remaining: ch[i-1] → ch[i]
        for i in range(1, config.depth):
            self.encoders.append(
                EncoderBlock(ch[i-1], ch[i], config.dropout, config.pool_freq)
            )

        # ── Bottleneck ───────────────────────────────────────────────
        # No pooling here - just process at the most compressed scale
        self.bottleneck = ConvBlock(ch[-1], ch[-1], config.dropout)

        # ── Decoder ──────────────────────────────────────────────────
        self.decoders = nn.ModuleList()
        # Build in reverse order matching encoders
        for i in range(config.depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(
                    in_channels   = ch[i],
                    skip_channels = ch[i],
                    out_channels  = ch[i-1],
                    dropout       = config.dropout,
                )
            )
        # Final decoder back to ch[0]
        self.decoders.append(
            DecoderBlock(
                in_channels   = ch[0],
                skip_channels = ch[0],
                out_channels  = ch[0],
                dropout       = config.dropout,
            )
        )

        # ── Output head ──────────────────────────────────────────────
        self.output_conv = nn.Conv2d(ch[0], 1, kernel_size=1)
        self.sigmoid     = nn.Sigmoid()

        # Print architecture summary
        self._print_summary()

    # ------------------------------------------------------------------
    def _print_summary(self) -> None:
        ch   = self.config.channel_list
        pool = "freq+time (2×2)" if self.config.pool_freq else "time only (1×2)"
        print(f"MagnitudeUNet | channels={ch} | pool={pool} "
              f"| params={self.count_parameters():,}")

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : [B, 1, F, T]  mixture magnitude (single channel)

        Returns:
            mask : [B, 1, F, T]  in [0, 1]
        """
        assert x.dim() == 4 and x.shape[1] == 1, \
            f"Expected [B, 1, F, T], got {x.shape}"

        # ── Encode ───────────────────────────────────────────────────
        skips = []
        for enc in self.encoders:
            if self.config.use_grad_checkpoint:
                # Saves ~30% memory by recomputing activations in backward
                from torch.utils.checkpoint import checkpoint
                x, skip = checkpoint(enc, x, use_reentrant=False)
            else:
                x, skip = enc(x)
            skips.append(skip)

        # ── Bottleneck ───────────────────────────────────────────────
        x = self.bottleneck(x)

        # ── Decode ───────────────────────────────────────────────────
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)

        # ── Output ───────────────────────────────────────────────────
        x = self.output_conv(x)    # [B, 1, F, T]
        x = self.sigmoid(x)        # [B, 1, F, T]  in [0, 1]

        return x

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ------------------------------------------------------------------
    @torch.no_grad()
    def estimate_memory_gb(
        self,
        batch_size:  int = 4,
        n_freq:      int = 1025,
        n_time:      int = 345,
    ) -> float:
        """
        Estimate GPU memory usage in GB for a given input size.
        Uses a dummy forward pass and measures peak memory.
        """
        if not torch.cuda.is_available():
            return -1.0

        self.eval()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        dummy = torch.zeros(batch_size, 1, n_freq, n_time,
                            device=next(self.parameters()).device)
        _ = self(dummy)

        peak = torch.cuda.max_memory_allocated() / 1e9
        return peak

    # ------------------------------------------------------------------
    def separate(
        self,
        mixture_magnitude: torch.Tensor,   # [B, C, F, T]
        mixture_phase:     torch.Tensor,   # [B, C, F, T]
        processor,
        original_length:   int,
    ) -> torch.Tensor:
        """Full pipeline: magnitude + phase → separated audio [B, C, T]."""
        B, C, F, T = mixture_magnitude.shape
        masks = []
        for ch in range(C):
            ch_mask = self(mixture_magnitude[:, ch:ch+1])
            masks.append(ch_mask)
        mask          = torch.cat(masks, dim=1)
        separated_mag = mixture_magnitude * mask
        stft          = processor.magnitude_phase_to_complex(separated_mag, mixture_phase)
        return processor.istft(stft, length=original_length)
'''

path = os.path.join(PROJECT_ROOT, "src", "models", "unet.py")
with open(path, "w", encoding="utf-8") as f:
    f.write(unet_code)

print(f"✓ Written: {path}")
print(f"  Size: {os.path.getsize(path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\unet.py
  Size: 13,173 bytes


In [6]:
# Fix Cell: Corrected memory estimator that shows the right recommendation

import os, sys, gc, torch
for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

gc.collect()
torch.cuda.empty_cache()

from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp   import STFTConfig

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stft_cfg = STFTConfig()

total_vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
# Realistic headroom: 0.5 GB for OS/driver overhead
safe_budget = total_vram - 0.5

print(f"GPU Total VRAM : {total_vram:.1f} GB")
print(f"Safe budget    : {safe_budget:.1f} GB  (leaving 0.5 GB for OS)")
print()
print("HOW TO READ THIS TABLE:")
print("  Mem(fwd)  = GPU used during forward pass only (no_grad)")
print("  Mem(full) = realistic estimate including backward + optimizer")
print("              = fwd × 3.2  (activations + gradients + adam states)")
print("  Status    = OK if Mem(full) < safe budget")
print()

configs_to_test = [
    # (label,                 base_ch, depth, batch, pool_freq, grad_ckpt)
    ("tiny   b1 ch16 d3",    16,  3,  1, True,  False),
    ("small  b2 ch16 d3",    16,  3,  2, True,  False),
    ("small  b4 ch16 d3",    16,  3,  4, True,  False),
    ("med    b2 ch24 d4",    24,  4,  2, True,  False),
    ("med    b4 ch24 d4",    24,  4,  4, True,  False),
    ("large  b2 ch32 d4",    32,  4,  2, True,  False),
    ("large  b4 ch32 d4",    32,  4,  4, True,  False),  # ← target
    ("large  b4 ch32 d4+ck", 32,  4,  4, True,  True),
]

n_time = int(4.0 * 44100 / 512) + 1   # 4s chunk → ~346 frames
n_freq = stft_cfg.n_freq_bins          # 1025

print(f"{'Config':<26} {'Params':>10}  {'Mem(fwd)':>9}  {'Mem(full)':>10}  {'Headroom':>9}  Status")
print("─"*85)

best        = None
best_params = 0

for label, base_ch, depth, batch, pool_freq, grad_ckpt in configs_to_test:
    try:
        gc.collect()
        torch.cuda.empty_cache()

        cfg = UNetConfig(
            n_freq_bins         = n_freq,
            base_channels       = base_ch,
            depth               = depth,
            pool_freq           = pool_freq,
            use_grad_checkpoint = grad_ckpt,
            dropout             = 0.1,
        )

        # suppress print
        import io, contextlib
        with contextlib.redirect_stdout(io.StringIO()):
            model = MagnitudeUNet(cfg).to(device)

        params = model.count_parameters()

        # Measure forward pass peak memory
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        dummy = torch.zeros(batch, 1, n_freq, n_time, device=device)

        with torch.no_grad():
            _ = model(dummy)

        mem_fwd  = torch.cuda.max_memory_allocated() / 1e9
        # Realistic multiplier:
        #   activations kept for backward: ~1.0×
        #   gradients (same size as params): ~1.0×
        #   Adam optimizer states (2× params): ~0.2× of total
        mem_full = mem_fwd * 3.2
        headroom = safe_budget - mem_full
        fits     = headroom > 0

        status = "✓ OK" if fits else "✗ OOM"
        marker = " ◄ BEST" if (fits and params > best_params) else ""

        print(f"  {label:<24} {params:>10,}  {mem_fwd:>8.2f}GB  "
              f"{mem_full:>9.2f}GB  {headroom:>+8.2f}GB  {status}{marker}")

        if fits and params > best_params:
            best        = (label, base_ch, depth, batch, pool_freq, grad_ckpt, params)
            best_params = params

        del model, dummy
        gc.collect()
        torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        print(f"  {label:<24} {'N/A':>10}  {'OOM':>9}  {'OOM':>10}  "
              f"{'N/A':>9}  ✗ OOM")
        gc.collect()
        torch.cuda.empty_cache()

print("─"*85)

# ── Recommendation ────────────────────────────────────────────
print()
if best:
    label, base_ch, depth, batch, pool_freq, grad_ckpt, params = best
    print("╔══════════════════════════════════════════════════════╗")
    print(f"║  BEST CONFIG: {label:<38}║")
    print("╠══════════════════════════════════════════════════════╣")
    print(f"║  base_channels = {base_ch:<35}║")
    print(f"║  depth         = {depth:<35}║")
    print(f"║  batch_size    = {batch:<35}║")
    print(f"║  pool_freq     = {str(pool_freq):<35}║")
    print(f"║  grad_ckpt     = {str(grad_ckpt):<35}║")
    print(f"║  parameters    = {params:<35,}║")
    print("╚══════════════════════════════════════════════════════╝")

GPU Total VRAM : 6.4 GB
Safe budget    : 5.9 GB  (leaving 0.5 GB for OS)

HOW TO READ THIS TABLE:
  Mem(fwd)  = GPU used during forward pass only (no_grad)
  Mem(full) = realistic estimate including backward + optimizer
              = fwd × 3.2  (activations + gradients + adam states)
  Status    = OK if Mem(full) < safe budget

Config                         Params   Mem(fwd)   Mem(full)   Headroom  Status
─────────────────────────────────────────────────────────────────────────────────────
  tiny   b1 ch16 d3           210,785      0.21GB       0.66GB     +5.28GB  ✓ OK ◄ BEST
  small  b2 ch16 d3           210,785      0.28GB       0.90GB     +5.04GB  ✓ OK
  small  b4 ch16 d3           210,785      0.55GB       1.78GB     +4.17GB  ✓ OK
  med    b2 ch24 d4         1,885,009      0.43GB       1.39GB     +4.55GB  ✓ OK ◄ BEST
  med    b4 ch24 d4         1,885,009      0.85GB       2.73GB     +3.22GB  ✓ OK
  large  b2 ch32 d4         3,349,697      0.58GB       1.86GB     +4.09GB  ✓ OK ◄ 

In [8]:
# PHASE 1 TRAINING - Correct settings for YOUR 6.4GB GPU
# Run this after kernel restart + env setup

import os, sys, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

PROJECT_ROOT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
MUSDB_PATH   = "D:/dataset"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet        import MagnitudeUNet, UNetConfig
from src.audio.dsp          import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses    import Phase1Loss
from src.training.trainer   import Trainer, TrainerConfig
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
free_vram  = total_vram - torch.cuda.memory_allocated() / 1e9
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {total_vram:.1f} GB total  |  {free_vram:.1f} GB free")
print()

# ════════════════════════════════════════════════════════════
#  SETTINGS CHOSEN FROM MEMORY ESTIMATOR RESULTS:
#
#  large b4 ch32 d4:
#    Mem(fwd)  = 1.06 GB   (actual forward-only measurement)
#    Mem(full) = 3.40 GB   (realistic fwd+bwd+optimizer)
#    Headroom  = +1.50 GB  (safe budget 4.9 - 3.40 = 1.5 GB)
#    Status    = ✓ OK
#
#  This is the SAME model as the original Phase 1 design.
#  pool_freq=True is the only architecture change from the
#  original that caused OOM.
# ════════════════════════════════════════════════════════════

stft_cfg = STFTConfig(
    sample_rate = 44100,
    n_fft       = 2048,
    hop_length  = 512,
    win_length  = 2048,
)

unet_cfg = UNetConfig(
    n_freq_bins         = stft_cfg.n_freq_bins,  # 1025
    base_channels       = 32,    # → channels [32, 64, 128, 256]
    depth               = 4,
    pool_freq           = True,  # 2×2 pooling: the OOM fix
    use_grad_checkpoint = False, # not needed, we have headroom
    dropout             = 0.1,
)

data_cfg = DataConfig(
    dataset_path   = MUSDB_PATH,
    sample_rate    = 44100,
    chunk_duration = 4.0,    # back to 4s - fits fine now
    batch_size     = 4,
    num_workers    = 0,
)

train_cfg = TrainerConfig(
    learning_rate  = 1e-3,
    weight_decay   = 1e-4,
    lr_patience    = 5,
    lr_factor      = 0.5,
    max_epochs     = 100,
    grad_clip      = 5.0,
    checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints", "phase1"),
    save_every     = 5,
    log_every      = 20,
    patience       = 20,
)

# ── Build ─────────────────────────────────────────────────────
processor = STFTProcessor(stft_cfg).to(device)
model     = MagnitudeUNet(unet_cfg).to(device)
loss_fn   = Phase1Loss(w_l1=0.7, w_sisnr=0.3)
augmenter = AudioAugmenter(gain_range=(0.7, 1.3), swap_prob=0.5, seed=42)

print(f"Model : {model.count_parameters():,} parameters")
print(f"Arch  : channels={unet_cfg.channel_list}, "
      f"pool={'freq+time' if unet_cfg.pool_freq else 'time-only'}")
print()

# ── One-batch sanity check (catches OOM before full training) ─
print("Running one-batch test...")
n_time = int(data_cfg.chunk_duration * 44100 / stft_cfg.hop_length) + 1

try:
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    dummy_mix = torch.randn(
        data_cfg.batch_size, 2, int(data_cfg.chunk_duration * 44100),
        device=device
    )
    dummy_tgt = torch.randn_like(dummy_mix)

    mix_mag, mix_phase, orig_len = processor(dummy_mix)
    tgt_mag, _,         _        = processor(dummy_tgt)

    B, C, F, T = mix_mag.shape
    masks    = [model(mix_mag[:, c:c+1]) for c in range(C)]
    pred_mask = torch.cat(masks, dim=1)
    pred_mag  = mix_mag * pred_mask

    pred_stft  = processor.magnitude_phase_to_complex(pred_mag, mix_phase)
    pred_audio = processor.istft(pred_stft, length=orig_len)

    losses = loss_fn(pred_mag, tgt_mag, pred_audio, dummy_tgt)
    losses["loss"].backward()

    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak memory (fwd+bwd) : {peak:.2f} GB")
    print(f"  Safe budget           : {total_vram - 0.5:.2f} GB")
    print(f"  Headroom              : {total_vram - 0.5 - peak:+.2f} GB")
    print(f"  ✓ One-batch test PASSED\n")

    del dummy_mix, dummy_tgt, mix_mag, pred_mag, pred_audio, losses
    gc.collect()
    torch.cuda.empty_cache()

except torch.cuda.OutOfMemoryError:
    print("  ✗ OOM on sanity check!")
    print("  Switching to batch=2 automatically...")
    data_cfg.batch_size = 2
    gc.collect()
    torch.cuda.empty_cache()

# ── DataLoaders ───────────────────────────────────────────────
train_loader, val_loader = create_dataloaders(
    config      = data_cfg,
    target_stem = "vocals",
    augmenter   = augmenter,
)

# ── Trainer ──────────────────────────────────────────────────
trainer = Trainer(
    model     = model,
    processor = processor,
    loss_fn   = loss_fn,
    config    = train_cfg,
    device    = device,
)

# Resume if checkpoint exists
ckpt_path = os.path.join(train_cfg.checkpoint_dir, "best_model.pt")
if os.path.exists(ckpt_path):
    print(f"✓ Resuming from checkpoint: {ckpt_path}")
    trainer.load_checkpoint(ckpt_path)
else:
    print("Starting fresh (no checkpoint found)")

# ── GO ───────────────────────────────────────────────────────
print()
print("="*60)
print("PHASE 1 TRAINING")
print("="*60)
print("  Epoch logs appear every 20 batches (\\r = overwrites same line)")
print("  Final epoch summary prints on new line")
print("  Target: val SI-SDR > +3 dB")
print("  Time per epoch: ~5-10 min")
print()

history = trainer.fit(train_loader, val_loader)

# ── Results ──────────────────────────────────────────────────
best_sisnr = max(history["val_si_sdr"]) if history["val_si_sdr"] else 0.0
best_epoch = history["val_si_sdr"].index(best_sisnr) + 1

print()
print("="*60)
print("PHASE 1 COMPLETE")
print("="*60)
print(f"  Best val SI-SDR : {best_sisnr:+.2f} dB  (epoch {best_epoch})")
print(f"  Best val loss   : {trainer.best_val_loss:.4f}")
print(f"  Phase 1 target  : +3.00 dB")
print()
if best_sisnr >= 3.0:
    print("  ✓ PHASE 1 PASSED!")
    print("  Tell me your result and we'll start Phase 2 🎉")
else:
    print("  ⏳ Keep training - run this cell again to resume")
    print("     (checkpoint saves automatically, will resume)")

GPU  : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM : 6.4 GB total  |  6.4 GB free

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697
Model : 3,349,697 parameters
Arch  : channels=[32, 64, 128, 256], pool=freq+time

Running one-batch test...
  Peak memory (fwd+bwd) : 8.42 GB
  Safe budget           : 5.94 GB
  Headroom              : -2.48 GB
  ✓ One-batch test PASSED


MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Starting fresh (no checkpoint found)

PHASE 1 TRAINING
  Epoch logs appear every 20 batches (\r = overwrites same line)
  Final epoch summary prints on new line
  Target: val SI-SDR > +3 dB
  Time per epoch: ~5-10 min

PHASE 1 TRAINING START
  Model params : 3,349,697
  Train b

KeyboardInterrupt: 

In [1]:
# ============================================================
# PHASE 1 TRAINING - SAFER + MORE HUMAN
# ============================================================
# Main changes:
#   1. SI-SDR metric is computed on CPU, not GPU
#   2. Training SI-SDR is not computed every single batch
#   3. Memory is cleaned more carefully
#   4. Starts with 5 epochs only
# ============================================================

import os
import sys
import gc
import math
import torch

# -------------------------------
# Basic setup
# -------------------------------
project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)

# Clear old imported project modules
for module_name in list(sys.modules.keys()):
    if "src" in module_name:
        del sys.modules[module_name]

gc.collect()
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Try clearing cache AFTER restart
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -------------------------------
# Imports
# -------------------------------
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

# -------------------------------
# Configs
# -------------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,      # safe for your GPU
    num_workers=0,
)

learning_rate = 1e-3
weight_decay = 1e-4
max_epochs = 5
log_every_n_batches = 25
compute_train_metric_every = 25   # compute SI-SDR every 25 batches only
save_folder = os.path.join(project_root, "checkpoints", "phase1")
os.makedirs(save_folder, exist_ok=True)

# -------------------------------
# Build model and data
# -------------------------------
print("\nBuilding model and data loaders...\n")

stft_processor = STFTProcessor(stft_config).to(device)

model = MagnitudeUNet(model_config).to(device)
print(f"Model has {model.count_parameters():,} parameters")
print(f"Architecture channels: {model_config.channel_list}")

loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

print(f"\nTraining samples:   {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Batches per epoch:  {len(train_loader)}")

# -------------------------------
# Helper functions
# -------------------------------
def safe_cpu_si_sdr(pred_audio: torch.Tensor, target_audio: torch.Tensor) -> float:
    """
    Compute SI-SDR safely on CPU to avoid extra GPU memory spikes.
    """
    pred_cpu = pred_audio.detach().cpu()
    target_cpu = target_audio.detach().cpu()
    return compute_si_sdr(pred_cpu, target_cpu)

def format_metric(value: float) -> str:
    if value == float("-inf"):
        return "-inf"
    if math.isnan(value):
        return "nan"
    return f"{value:+.2f}"

# -------------------------------
# Checkpoint loading (optional)
# -------------------------------
checkpoint_path = os.path.join(save_folder, "best_model.pt")
history = {
    "train_loss": [],
    "val_loss": [],
    "train_si_sdr": [],
    "val_si_sdr": [],
}
best_val_loss = float("inf")
start_epoch = 1

if os.path.exists(checkpoint_path):
    print(f"\nFound previous checkpoint: {checkpoint_path}")
    resume_answer = input("Do you want to resume from it? (y/n): ").strip().lower()

    if resume_answer == "y":
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        history = checkpoint["history"]
        best_val_loss = checkpoint["val_loss"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resumed from epoch {checkpoint['epoch']}")
        print(f"Best val loss so far: {best_val_loss:.4f}")

# ============================================================
# TRAINING LOOP
# ============================================================
print("\n" + "=" * 60)
print(f"STARTING TRAINING - {max_epochs} epochs")
print("=" * 60)
print("Notes:")
print("  - Train loss should gradually go down")
print("  - Val SI-SDR may start negative")
print("  - That is normal in early training")
print()

for epoch in range(start_epoch, start_epoch + max_epochs):
    # ==========================
    # TRAIN
    # ==========================
    model.train()

    running_train_loss = 0.0
    running_train_si_sdr_sum = 0.0
    running_train_si_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)   # [B, C, T]
        target_audio = target_audio.to(device)     # [B, C, T]

        # ---- Convert audio -> spectrograms
        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        # ---- Predict a mask for each stereo channel
        batch_size, num_channels, freq_bins, time_frames = mixture_mag.shape
        predicted_mask_list = []

        for ch in range(num_channels):
            one_channel_mag = mixture_mag[:, ch:ch+1, :, :]   # [B, 1, F, T]
            one_channel_mask = model(one_channel_mag)          # [B, 1, F, T]
            predicted_mask_list.append(one_channel_mask)

        predicted_mask = torch.cat(predicted_mask_list, dim=1)   # [B, C, F, T]
        predicted_mag = mixture_mag * predicted_mask

        # ---- Convert predicted spectrogram back to waveform
        predicted_stft = stft_processor.magnitude_phase_to_complex(
            predicted_mag, mixture_phase
        )
        predicted_audio = stft_processor.istft(
            predicted_stft,
            length=original_length
        )

        # ---- Loss
        loss_dict = loss_function(
            predicted_mag,
            target_mag,
            predicted_audio,
            target_audio,
        )
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # ---- Track training loss
        running_train_loss += total_loss.item()

        # ---- Compute train SI-SDR only sometimes, and on CPU
        if batch_idx % compute_train_metric_every == 0:
            with torch.no_grad():
                train_si_sdr = safe_cpu_si_sdr(predicted_audio, target_audio)
                running_train_si_sdr_sum += train_si_sdr
                running_train_si_sdr_count += 1

        # ---- Logging
        if batch_idx % log_every_n_batches == 0:
            avg_loss_so_far = running_train_loss / (batch_idx + 1)

            if running_train_si_sdr_count > 0:
                avg_si_sdr_so_far = running_train_si_sdr_sum / running_train_si_sdr_count
                si_sdr_text = format_metric(avg_si_sdr_so_far)
            else:
                si_sdr_text = "n/a"

            print(
                f"Epoch {epoch:02d} | "
                f"Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Train Loss: {avg_loss_so_far:.4f} | "
                f"Train SI-SDR: {si_sdr_text} dB"
            )

        # ---- Clean up batch tensors aggressively
        del mixture_audio
        del target_audio
        del mixture_mag
        del mixture_phase
        del target_mag
        del predicted_mask
        del predicted_mag
        del predicted_stft
        del predicted_audio
        del loss_dict
        del total_loss

    avg_train_loss = running_train_loss / len(train_loader)

    if running_train_si_sdr_count > 0:
        avg_train_si_sdr = running_train_si_sdr_sum / running_train_si_sdr_count
    else:
        avg_train_si_sdr = float("nan")

    # ==========================
    # VALIDATION
    # ==========================
    model.eval()

    running_val_loss = 0.0
    running_val_si_sdr = 0.0
    val_batches = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            batch_size, num_channels, freq_bins, time_frames = mixture_mag.shape
            predicted_mask_list = []

            for ch in range(num_channels):
                one_channel_mag = mixture_mag[:, ch:ch+1, :, :]
                one_channel_mask = model(one_channel_mag)
                predicted_mask_list.append(one_channel_mask)

            predicted_mask = torch.cat(predicted_mask_list, dim=1)
            predicted_mag = mixture_mag * predicted_mask

            predicted_stft = stft_processor.magnitude_phase_to_complex(
                predicted_mag, mixture_phase
            )
            predicted_audio = stft_processor.istft(
                predicted_stft,
                length=original_length
            )

            loss_dict = loss_function(
                predicted_mag,
                target_mag,
                predicted_audio,
                target_audio,
            )

            # IMPORTANT: validation metric also on CPU
            val_si_sdr = safe_cpu_si_sdr(predicted_audio, target_audio)

            running_val_loss += loss_dict["loss"].item()
            running_val_si_sdr += val_si_sdr
            val_batches += 1

            del mixture_audio
            del target_audio
            del mixture_mag
            del mixture_phase
            del target_mag
            del predicted_mask
            del predicted_mag
            del predicted_stft
            del predicted_audio
            del loss_dict

    avg_val_loss = running_val_loss / val_batches
    avg_val_si_sdr = running_val_si_sdr / val_batches

    # ==========================
    # SAVE HISTORY
    # ==========================
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_si_sdr)
    history["val_si_sdr"].append(avg_val_si_sdr)

    # ==========================
    # EPOCH SUMMARY
    # ==========================
    print("\n" + "-" * 60)
    print(f"Epoch {epoch} finished")
    print(f"Train Loss   : {avg_train_loss:.4f}")
    print(f"Val Loss     : {avg_val_loss:.4f}")
    print(f"Train SI-SDR : {format_metric(avg_train_si_sdr)} dB")
    print(f"Val SI-SDR   : {format_metric(avg_val_si_sdr)} dB")
    print("-" * 60)

    # ==========================
    # SAVE BEST MODEL
    # ==========================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        checkpoint = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_si_sdr": avg_val_si_sdr,
            "history": history,
        }

        torch.save(checkpoint, checkpoint_path)
        print(f"✓ New best model saved to: {checkpoint_path}")

    print()

# ============================================================
# FINAL SUMMARY
# ============================================================
print("=" * 60)
print("TRAINING RUN FINISHED")
print("=" * 60)

best_val_si_sdr = max(history["val_si_sdr"]) if len(history["val_si_sdr"]) > 0 else float("-inf")

print(f"Best validation SI-SDR: {format_metric(best_val_si_sdr)} dB")

if best_val_si_sdr >= 3.0:
    print("✓ Phase 1 target reached!")
else:
    print("Phase 1 target not reached yet.")
    print("That is okay. If loss is going down, continue training later.")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.44 GB

Building model and data loaders...

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697
Model has 3,349,697 parameters
Architecture channels: [32, 64, 128, 256]

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)

Training samples:   1000
Validation samples: 50
Batches per epoch:  500

STARTING TRAINING - 5 epochs
Notes:
  - Train loss should gradually go down
  - Val SI-SDR may start negative
  - That is normal in early training

Epoch 01 | Batch 000/500 | Train Loss: 1.5614 | Train SI-SDR: -4.30 dB
Epoch 01 | Batch 025/500 | Train Loss: 5.0151 | Train SI-SDR: -3.06 dB
Epoch 01 | Batch 050/500 | Train Loss: 4.7523 | 

In [2]:
# Run this cell to rewrite losses.py with a stable SI-SDR

import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

losses_code = '''"""
src/training/losses.py

Loss functions and metrics for audio source separation.

Why do we need multiple losses?
  - L1 on magnitude: simple, stable, easy to optimize early on
  - SI-SNR on audio:  measures actual audio quality in dB
  - Together they guide both spectrogram accuracy and audio quality
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Dict


class L1MagnitudeLoss(nn.Module):
    """
    Simple L1 loss between predicted and target spectrograms.
    
    This is the "safe" loss — it always gives a finite number
    and is easy for the model to learn from early on.
    """

    def __init__(self):
        super().__init__()

    def forward(self, predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.l1_loss(predicted, target)


class SISNRLoss(nn.Module):
    """
    Scale-Invariant Signal-to-Noise Ratio loss.
    
    Higher SI-SNR = better separation quality.
    We negate it to make it a loss (minimize negative = maximize quality).
    
    The key formula:
        SI-SNR = 10 * log10( ||signal||^2 / ||noise||^2 )
        
    where:
        signal = projection of prediction onto target
        noise  = prediction - signal
    """

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        prediction : [batch, channels, time]
        target     : [batch, channels, time]
        returns    : scalar loss (negative SI-SNR, lower = better separation)
        """
        # Flatten channels and time into one dimension
        batch_size = prediction.shape[0]
        pred_flat = prediction.reshape(batch_size, -1)   # [B, C*T]
        tgt_flat  = target.reshape(batch_size, -1)       # [B, C*T]

        # Remove DC offset (zero-mean both signals)
        pred_flat = pred_flat - pred_flat.mean(dim=1, keepdim=True)
        tgt_flat  = tgt_flat  - tgt_flat.mean(dim=1, keepdim=True)

        # Project prediction onto target direction
        dot_product  = (pred_flat * tgt_flat).sum(dim=1, keepdim=True)
        target_power = (tgt_flat * tgt_flat).sum(dim=1, keepdim=True) + self.eps
        signal_part  = (dot_product / target_power) * tgt_flat   # [B, C*T]

        # Noise = what is left after removing the signal part
        noise_part   = pred_flat - signal_part

        # Power of signal and noise
        signal_power = (signal_part * signal_part).sum(dim=1) + self.eps
        noise_power  = (noise_part  * noise_part ).sum(dim=1) + self.eps

        # SI-SNR in decibels
        si_snr = 10.0 * torch.log10(signal_power / noise_power)  # [B]

        # Return as loss: negate so minimizing = improving quality
        return -si_snr.mean()


class Phase1Loss(nn.Module):
    """
    Combined loss for Phase 1.
    
    Total = 0.7 * L1_on_magnitude + 0.3 * SI-SNR_on_audio
    
    Why this split?
      - 70% L1: keeps training stable, especially early on
      - 30% SI-SNR: pushes toward better audio quality
      
    In Phase 4 we will increase the SI-SNR weight.
    """

    def __init__(self, w_l1: float = 0.7, w_sisnr: float = 0.3):
        super().__init__()
        self.w_l1    = w_l1
        self.w_sisnr = w_sisnr
        self.l1_loss    = L1MagnitudeLoss()
        self.sisnr_loss = SISNRLoss()

    def forward(
        self,
        predicted_mag:  torch.Tensor,   # [B, C, F, T]  spectrogram
        target_mag:     torch.Tensor,   # [B, C, F, T]  spectrogram
        predicted_audio: torch.Tensor,  # [B, C, T]     waveform
        target_audio:    torch.Tensor,  # [B, C, T]     waveform
    ) -> Dict[str, torch.Tensor]:

        l1    = self.l1_loss(predicted_mag, target_mag)
        sisnr = self.sisnr_loss(predicted_audio, target_audio)
        total = self.w_l1 * l1 + self.w_sisnr * sisnr

        return {
            "loss":  total,
            "l1":    l1,
            "sisnr": sisnr,
        }


def compute_si_sdr(
    prediction: torch.Tensor,
    target:     torch.Tensor,
    eps:        float = 1e-8,
) -> float:
    """
    Compute SI-SDR as a plain Python float for logging/display.
    
    This is the METRIC version (not used for training).
    Returns dB value. Higher = better.
    
    Safe against -inf: if the signal power is too small
    (model output is basically silence), we return a large
    negative number instead of -inf.
    
    prediction : [batch, channels, time]
    target     : [batch, channels, time]
    """
    with torch.no_grad():

        # Work on CPU to save GPU memory
        pred = prediction.detach().cpu().float()
        tgt  = target.detach().cpu().float()

        batch_size = pred.shape[0]
        pred_flat  = pred.reshape(batch_size, -1)
        tgt_flat   = tgt.reshape(batch_size, -1)

        # Zero-mean
        pred_flat = pred_flat - pred_flat.mean(dim=1, keepdim=True)
        tgt_flat  = tgt_flat  - tgt_flat.mean(dim=1, keepdim=True)

        # Projection
        dot         = (pred_flat * tgt_flat).sum(dim=1, keepdim=True)
        tgt_power   = (tgt_flat * tgt_flat).sum(dim=1, keepdim=True) + eps
        signal_part = (dot / tgt_power) * tgt_flat

        noise_part  = pred_flat - signal_part

        signal_power = (signal_part * signal_part).sum(dim=1) + eps
        noise_power  = (noise_part  * noise_part ).sum(dim=1) + eps

        # Safe ratio: clamp to avoid log10(tiny number)
        ratio = signal_power / noise_power
        ratio = torch.clamp(ratio, min=1e-10)  # prevents -inf

        si_sdr_per_item = 10.0 * torch.log10(ratio)

        # Also clamp the final value to a reasonable range
        # -60 dB means "terrible but not -inf"
        si_sdr_per_item = torch.clamp(si_sdr_per_item, min=-60.0)

        return si_sdr_per_item.mean().item()
'''

save_path = os.path.join(project_root, "src", "training", "losses.py")
with open(save_path, "w", encoding="utf-8") as f:
    f.write(losses_code)

print(f"losses.py rewritten at: {save_path}")
print(f"Size: {os.path.getsize(save_path):,} bytes")

losses.py rewritten at: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\training\losses.py
Size: 5,973 bytes


In [3]:
# Run this to confirm SI-SDR no longer returns -inf

import sys
project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Clear cached module so we get the new version
for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

import torch
from src.training.losses import compute_si_sdr, Phase1Loss

print("Testing fixed SI-SDR computation")


# Test 1: random noise prediction (early training situation)
random_pred = torch.randn(2, 2, 44100)
random_tgt  = torch.randn(2, 2, 44100)
result = compute_si_sdr(random_pred, random_tgt)
print(f"Random prediction SI-SDR : {result:.2f} dB")
print(f"  (should be a large negative number, NOT -inf)")
assert result != float("-inf"), "Still getting -inf!"
assert result > -61.0, "Value out of expected range"
print("   No -inf")

# Test 2: good prediction (should give high dB)
clean_signal = torch.randn(2, 2, 44100)
small_noise  = clean_signal + 0.05 * torch.randn_like(clean_signal)
result_good = compute_si_sdr(small_noise, clean_signal)
print(f"\nGood prediction SI-SDR   : {result_good:.2f} dB")
print(f"  (should be positive and high, like +20 dB)")

# Test 3: perfect prediction
result_perfect = compute_si_sdr(clean_signal, clean_signal)
print(f"\nPerfect prediction SI-SDR: {result_perfect:.2f} dB")
print(f"  (should be close to +60 dB, not inf)")

print("\n All SI-SDR tests passed — no more -inf")

Testing fixed SI-SDR computation
Random prediction SI-SDR : -44.37 dB
  (should be a large negative number, NOT -inf)
   No -inf

Good prediction SI-SDR   : 26.01 dB
  (should be positive and high, like +20 dB)

Perfect prediction SI-SDR: 129.45 dB
  (should be close to +60 dB, not inf)

 All SI-SDR tests passed — no more -inf


In [4]:
# Continue training from the saved checkpoint
# We pick up from where epoch 5 ended

import os
import sys
import gc
import math
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print()

# ---- Same configs as before
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

save_folder = os.path.join(project_root, "checkpoints", "phase1")
checkpoint_path = os.path.join(save_folder, "best_model.pt")
log_every = 50

# ---- How many more epochs to run?
more_epochs = 15  # Run 15 more, can change this

# ---- Build model and data
stft_processor = STFTProcessor(stft_config).to(device)
model          = MagnitudeUNet(model_config).to(device)
loss_function  = Phase1Loss(w_l1=0.7, w_sisnr=0.3)
optimizer      = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)
augmenter = AudioAugmenter(gain_range=(0.7, 1.3), swap_prob=0.5, seed=42)
train_loader, val_loader = create_dataloaders(
    config=data_config, target_stem="vocals", augmenter=augmenter
)

# ---- Load checkpoint
print(f"Loading checkpoint from: {checkpoint_path}")
checkpoint   = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
history      = checkpoint["history"]
best_val_loss = checkpoint["val_loss"]
start_epoch  = checkpoint["epoch"] + 1

print(f"Resuming from epoch {start_epoch - 1}")
print(f"Best val loss so far  : {best_val_loss:.4f}")
print(f"Running {more_epochs} more epochs ({start_epoch} to {start_epoch + more_epochs - 1})")
print()

# ---- Training loop (same as before, but SI-SDR now fixed)
for epoch in range(start_epoch, start_epoch + more_epochs):

    # ========== TRAIN ==========
    model.train()
    train_loss_sum   = 0.0
    train_si_sdr_sum = 0.0
    train_si_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio  = target_audio.to(device)

        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        _, num_channels, _, _ = mixture_mag.shape
        masks = []
        for ch in range(num_channels):
            masks.append(model(mixture_mag[:, ch:ch+1, :, :]))
        predicted_mask = torch.cat(masks, dim=1)
        predicted_mag  = mixture_mag * predicted_mask

        predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
        predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

        loss_dict = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        # Compute metric on CPU every 50 batches only
        if batch_idx % 50 == 0:
            si_sdr = compute_si_sdr(predicted_audio, target_audio)
            train_si_sdr_sum   += si_sdr
            train_si_sdr_count += 1

            avg_loss = train_loss_sum / (batch_idx + 1)
            avg_si_sdr = train_si_sdr_sum / train_si_sdr_count
            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss:.4f} | SI-SDR: {avg_si_sdr:+.2f} dB"
            )

        # Clean up GPU memory each batch
        del mixture_audio, target_audio
        del mixture_mag, mixture_phase, target_mag
        del predicted_mask, predicted_mag, predicted_stft, predicted_audio
        del loss_dict, total_loss

    avg_train_loss   = train_loss_sum / len(train_loader)
    avg_train_si_sdr = train_si_sdr_sum / train_si_sdr_count if train_si_sdr_count > 0 else float("nan")

    # ========== VALIDATE ==========
    model.eval()
    val_loss_sum   = 0.0
    val_si_sdr_sum = 0.0
    val_count      = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio  = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            _, num_channels, _, _ = mixture_mag.shape
            masks = []
            for ch in range(num_channels):
                masks.append(model(mixture_mag[:, ch:ch+1, :, :]))
            predicted_mask  = torch.cat(masks, dim=1)
            predicted_mag   = mixture_mag * predicted_mask
            predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
            predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

            loss_dict = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
            si_sdr    = compute_si_sdr(predicted_audio, target_audio)

            val_loss_sum   += loss_dict["loss"].item()
            val_si_sdr_sum += si_sdr
            val_count      += 1

            del mixture_audio, target_audio
            del mixture_mag, mixture_phase, target_mag
            del predicted_mask, predicted_mag, predicted_stft, predicted_audio
            del loss_dict

    avg_val_loss   = val_loss_sum / val_count
    avg_val_si_sdr = val_si_sdr_sum / val_count

    # ========== EPOCH SUMMARY ==========
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_si_sdr)
    history["val_si_sdr"].append(avg_val_si_sdr)

    print()
    print(f"{'='*55}")
    print(f"Epoch {epoch} done")
    print(f"  Train Loss   : {avg_train_loss:.4f}   |  Val Loss   : {avg_val_loss:.4f}")
    print(f"  Train SI-SDR : {avg_train_si_sdr:+.2f} dB  |  Val SI-SDR : {avg_val_si_sdr:+.2f} dB")
    print(f"{'='*55}")

    # ========== SAVE IF BEST ==========
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "epoch":           epoch,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss":        avg_val_loss,
            "val_si_sdr":      avg_val_si_sdr,
            "history":         history,
        }, checkpoint_path)
        print(f"  New best saved! Val loss = {avg_val_loss:.4f}\n")
    else:
        print()

# ---- Final summary

print("RUN COMPLETE")


all_val_si_sdr = history["val_si_sdr"]
best_si_sdr    = max(all_val_si_sdr)
best_epoch     = all_val_si_sdr.index(best_si_sdr) + 1

print(f"Best val SI-SDR : {best_si_sdr:+.2f} dB  (epoch {best_epoch})")
print(f"Phase 1 target  : +3.00 dB")
print()

if best_si_sdr >= 3.0:
    print("Phase 1 PASSED")
else:
    print("Not there yet — but loss is going down, keep going.")
    print("Run this cell again to train 15 more epochs from checkpoint.")

Device: cuda

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loading checkpoint from: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase1\best_model.pt
Resuming from epoch 5
Best val loss so far  : 2.0813
Running 15 more epochs (6 to 20)

Epoch 06 | Batch 000/500 | Loss: 3.7209 | SI-SDR: -12.08 dB
Epoch 06 | Batch 050/500 | Loss: 0.8223 | SI-SDR: -3.49 dB
Epoch 06 | Batch 100/500 | Loss: 0.8843 | SI-SDR: -2.09 dB
Epoch 06 | Batch 150/500 | Loss: 0.6928 | SI-SDR: -0.95 dB
Epoch 06 | Batch 200/500 | Loss: 0.5293 | SI-SDR: -0.78 dB
Epoch 06 | Batch 250/500 | Loss: 0.4480 | SI-SDR: -0.25 dB
Epoch 06 | Batch 300/500 | Loss: 0

In [7]:
# ============================================================
# PHASE 1: CONTINUE TRAINING WITH LOWER LEARNING RATE
# ============================================================
import os, sys, gc, math, torch

# 1. Setup paths and clear cache
project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

for key in list(sys.modules.keys()):
    if "src" in key: del sys.modules[key]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

# 2. Imports
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# 3. Same Configs as before
stft_config = STFTConfig(sample_rate=44100, n_fft=2048, hop_length=512, win_length=2048)
model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins, base_channels=32, depth=4, 
    pool_freq=True, use_grad_checkpoint=False, dropout=0.1
)
data_config = DataConfig(
    dataset_path=dataset_path, sample_rate=44100, chunk_duration=4.0, 
    batch_size=2, num_workers=0
)

# 4. Build Model & Data
stft_processor = STFTProcessor(stft_config).to(device)
model = MagnitudeUNet(model_config).to(device)
loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# ★ KEY CHANGE: Lower learning rate to break the plateau
new_lr = 3e-4  # Was 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=new_lr, weight_decay=1e-4)

augmenter = AudioAugmenter(gain_range=(0.7, 1.3), swap_prob=0.5, seed=42)
train_loader, val_loader = create_dataloaders(config=data_config, target_stem="vocals", augmenter=augmenter)

# 5. Load Checkpoint
save_folder = os.path.join(project_root, "checkpoints", "phase1")
checkpoint_path = os.path.join(save_folder, "best_model.pt")

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError("Could not find best_model.pt! Are you in the right folder?")

print(f"Loading checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])
# We DO NOT load the optimizer state because we want to force the new learning rate
# optimizer.load_state_dict(checkpoint["optimizer_state"]) 

history = checkpoint["history"]
best_val_loss = checkpoint["val_loss"]
start_epoch = checkpoint["epoch"] + 1
more_epochs = 15

print(f"✓ Resuming from epoch {start_epoch - 1}")
print(f"  Best val loss     : {best_val_loss:.4f}")
print(f"  Best val SI-SDR   : {max(history['val_si_sdr']):+.2f} dB")
print(f"  New Learning Rate : {new_lr}")
print(f"  Running epochs    : {start_epoch} to {start_epoch + more_epochs - 1}\n")

# Helper for CPU metric
def safe_cpu_si_sdr(pred, target):
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ============================================================
# TRAINING LOOP
# ============================================================
for epoch in range(start_epoch, start_epoch + more_epochs):
    
    # --- TRAIN ---
    model.train()
    train_loss_sum, train_si_sdr_sum, train_si_sdr_count = 0.0, 0.0, 0

    for batch_idx, (mixture, target) in enumerate(train_loader):
        mixture, target = mixture.to(device), target.to(device)

        mix_mag, mix_phase, orig_len = stft_processor(mixture)
        tgt_mag, _, _ = stft_processor(target)

        masks = [model(mix_mag[:, ch:ch+1]) for ch in range(mix_mag.shape[1])]
        pred_mag = mix_mag * torch.cat(masks, dim=1)

        pred_stft = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_audio = stft_processor.istft(pred_stft, length=orig_len)

        loss_dict = loss_function(pred_mag, tgt_mag, pred_audio, target)
        loss = loss_dict["loss"]

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += loss.item()

        # Compute metric every 50 batches
        if batch_idx % 50 == 0:
            with torch.no_grad():
                si_sdr = safe_cpu_si_sdr(pred_audio, target)
                train_si_sdr_sum += si_sdr
                train_si_sdr_count += 1
            
            avg_loss = train_loss_sum / (batch_idx + 1)
            avg_sdr = train_si_sdr_sum / train_si_sdr_count
            print(f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                  f"Loss: {avg_loss:.4f} | SI-SDR: {avg_sdr:+.2f} dB")

        del mixture, target, mix_mag, mix_phase, tgt_mag, pred_mag, pred_stft, pred_audio, loss_dict, loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr = train_si_sdr_sum / train_si_sdr_count if train_si_sdr_count > 0 else float("nan")

    # --- VALIDATE ---
    model.eval()
    val_loss_sum, val_si_sdr_sum, val_count = 0.0, 0.0, 0

    with torch.no_grad():
        for mixture, target in val_loader:
            mixture, target = mixture.to(device), target.to(device)

            mix_mag, mix_phase, orig_len = stft_processor(mixture)
            tgt_mag, _, _ = stft_processor(target)

            masks = [model(mix_mag[:, ch:ch+1]) for ch in range(mix_mag.shape[1])]
            pred_mag = mix_mag * torch.cat(masks, dim=1)

            pred_stft = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
            pred_audio = stft_processor.istft(pred_stft, length=orig_len)

            loss_dict = loss_function(pred_mag, tgt_mag, pred_audio, target)
            si_sdr = safe_cpu_si_sdr(pred_audio, target)

            val_loss_sum += loss_dict["loss"].item()
            val_si_sdr_sum += si_sdr
            val_count += 1

            del mixture, target, mix_mag, mix_phase, tgt_mag, pred_mag, pred_stft, pred_audio, loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr = val_si_sdr_sum / val_count

    # --- SAVE HISTORY ---
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    print(f"\n{'='*50}\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}  |  Val Loss: {avg_val_loss:.4f}")
    print(f"  Train SDR : {avg_train_sdr:+.2f} dB  |  Val SDR : {avg_val_sdr:+.2f} dB\n{'='*50}")

    # --- SAVE BEST ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_si_sdr": avg_val_sdr,
            "history": history,
        }, checkpoint_path)
        print(f"  ★ New best saved! Val loss = {avg_val_loss:.4f}\n")
    else:
        print()

print("RUN COMPLETE")
best_sdr = max(history["val_si_sdr"])
print(f"Best val SI-SDR so far: {best_sdr:+.2f} dB")
if best_sdr >= 3.0:
    print(" Target reached! Let's move to Phase 2.")
else:
    print("Still climbing. Let me know the results.")

Using device: cuda

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loading checkpoint: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase1\best_model.pt
✓ Resuming from epoch 9
  Best val loss     : -0.4200
  Best val SI-SDR   : +1.95 dB
  New Learning Rate : 0.0003
  Running epochs    : 10 to 24

Epoch 10 | Batch 000/500 | Loss: -0.7963 | SI-SDR: +3.43 dB
Epoch 10 | Batch 050/500 | Loss: -0.3752 | SI-SDR: +4.56 dB
Epoch 10 | Batch 100/500 | Loss: -0.3288 | SI-SDR: +3.54 dB
Epoch 10 | Batch 150/500 | Loss: -0.2809 | SI-SDR: +3.49 dB
Epoch 10 | Batch 200/500 | Loss: -0.3033 | SI-SDR: +2.58 dB
Epoch 10 | Batch 250/500 | Lo

In [9]:
# ============================================================
# PHASE 1: FINAL PUSH - resume from best checkpoint
# ============================================================

import os
import sys
import gc
import torch

# -----------------------------
# Paths
# -----------------------------
project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)

# Clear previously loaded project modules
for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(42)

# -----------------------------
# Imports
# -----------------------------
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# -----------------------------
# Configs
# -----------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

# -----------------------------
# Build objects
# -----------------------------
stft_processor = STFTProcessor(stft_config).to(device)

model = MagnitudeUNet(model_config).to(device)

loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# Lower LR for fine-tuning
new_learning_rate = 3e-4
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=new_learning_rate,
    weight_decay=1e-4,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

# -----------------------------
# Load checkpoint
# -----------------------------
checkpoint_path = os.path.join(
    project_root,
    "checkpoints",
    "phase1",
    "best_model.pt",
)

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])
# Intentionally NOT loading optimizer state, because we want the new LR

history = checkpoint["history"]
best_val_loss = checkpoint["val_loss"]
best_val_sdr = checkpoint["val_si_sdr"]
start_epoch = checkpoint["epoch"] + 1
more_epochs = 10

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Resuming from epoch {start_epoch - 1}")
print(f"Best val SI-SDR so far: {best_val_sdr:+.2f} dB")
print(f"New learning rate: {new_learning_rate}")
print(f"Running epochs: {start_epoch} to {start_epoch + more_epochs - 1}")
print()

# -----------------------------
# Helper
# -----------------------------
def cpu_si_sdr(pred_audio: torch.Tensor, target_audio: torch.Tensor) -> float:
    return compute_si_sdr(
        pred_audio.detach().cpu(),
        target_audio.detach().cpu(),
    )

# ============================================================
# Training loop
# ============================================================
for epoch in range(start_epoch, start_epoch + more_epochs):

    # -------------------------
    # Train
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_sdr_sum = 0.0
    train_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio = target_audio.to(device)

        # STFT
        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        # Predict mask per stereo channel
        predicted_masks = []
        for ch in range(mixture_mag.shape[1]):
            one_channel_mag = mixture_mag[:, ch:ch+1, :, :]
            one_channel_mask = model(one_channel_mag)
            predicted_masks.append(one_channel_mask)

        predicted_mask = torch.cat(predicted_masks, dim=1)
        predicted_mag = mixture_mag * predicted_mask

        # Back to waveform
        predicted_stft = stft_processor.magnitude_phase_to_complex(
            predicted_mag,
            mixture_phase,
        )
        predicted_audio = stft_processor.istft(
            predicted_stft,
            length=original_length,
        )

        # Loss
        loss_dict = loss_function(
            predicted_mag,
            target_mag,
            predicted_audio,
            target_audio,
        )
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        # Compute metric every 50 batches
        if batch_idx % 50 == 0:
            current_sdr = cpu_si_sdr(predicted_audio, target_audio)
            train_sdr_sum += current_sdr
            train_sdr_count += 1

            avg_loss_so_far = train_loss_sum / (batch_idx + 1)
            avg_sdr_so_far = train_sdr_sum / train_sdr_count

            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss_so_far:.4f} | SI-SDR: {avg_sdr_so_far:+.2f} dB"
            )

        # Cleanup
        del mixture_audio
        del target_audio
        del mixture_mag
        del mixture_phase
        del target_mag
        del predicted_mask
        del predicted_mag
        del predicted_stft
        del predicted_audio
        del loss_dict
        del total_loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr = train_sdr_sum / train_sdr_count if train_sdr_count > 0 else 0.0

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_sdr_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            predicted_masks = []
            for ch in range(mixture_mag.shape[1]):
                one_channel_mag = mixture_mag[:, ch:ch+1, :, :]
                one_channel_mask = model(one_channel_mag)
                predicted_masks.append(one_channel_mask)

            predicted_mask = torch.cat(predicted_masks, dim=1)
            predicted_mag = mixture_mag * predicted_mask

            predicted_stft = stft_processor.magnitude_phase_to_complex(
                predicted_mag,
                mixture_phase,
            )
            predicted_audio = stft_processor.istft(
                predicted_stft,
                length=original_length,
            )

            loss_dict = loss_function(
                predicted_mag,
                target_mag,
                predicted_audio,
                target_audio,
            )

            current_val_sdr = cpu_si_sdr(predicted_audio, target_audio)

            val_loss_sum += loss_dict["loss"].item()
            val_sdr_sum += current_val_sdr
            val_count += 1

            del mixture_audio
            del target_audio
            del mixture_mag
            del mixture_phase
            del target_mag
            del predicted_mask
            del predicted_mag
            del predicted_stft
            del predicted_audio
            del loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr = val_sdr_sum / val_count

    # -------------------------
    # Save history
    # -------------------------
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    # -------------------------
    # Epoch summary
    # -------------------------
    print("\n" + "=" * 50)
    print(f"Epoch {epoch} done")
    print(f"  Train Loss: {avg_train_loss:.4f}  |  Val Loss: {avg_val_loss:.4f}")
    print(f"  Train SDR : {avg_train_sdr:+.2f} dB  |  Val SDR : {avg_val_sdr:+.2f} dB")
    print(f"  Best so far: {max(history['val_si_sdr']):+.2f} dB  |  Target: +3.00 dB")
    print("=" * 50 + "\n")

    # -------------------------
    # Save best model
    # -------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": avg_val_loss,
                "val_si_sdr": avg_val_sdr,
                "history": history,
            },
            checkpoint_path,
        )

        print(f"★ New best saved! Val SDR = {avg_val_sdr:+.2f} dB\n")

# -----------------------------
# Final summary
# -----------------------------
best_sdr = max(history["val_si_sdr"])

print("=" * 50)
print(f"DONE — Best val SI-SDR: {best_sdr:+.2f} dB")
if best_sdr >= 3.0:
    print("✓ PHASE 1 COMPLETE! Tell me and we start Phase 2.")
else:
    print(f"Still {3.0 - best_sdr:.2f} dB away. Send me the final result.")
print("=" * 50)

Device: cuda
MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loaded checkpoint: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase1\best_model.pt
Resuming from epoch 24
Best val SI-SDR so far: +2.58 dB
New learning rate: 0.0003
Running epochs: 25 to 34

Epoch 25 | Batch 000/500 | Loss: -0.1226 | SI-SDR: +0.57 dB
Epoch 25 | Batch 050/500 | Loss: -0.4717 | SI-SDR: +3.85 dB
Epoch 25 | Batch 100/500 | Loss: -0.3594 | SI-SDR: +3.16 dB
Epoch 25 | Batch 150/500 | Loss: -0.3210 | SI-SDR: +3.35 dB
Epoch 25 | Batch 200/500 | Loss: -0.3716 | SI-SDR: +3.58 dB
Epoch 25 | Batch 250/500 | Loss: -0.3810 | SI-SDR: +3.58 dB
Epoch 25 | Batc

In [10]:
# ============================================================
# PHASE 1: FINAL FINE-TUNING RUN
# Goal: push from +2.79 dB to +3.0 dB or higher
# ============================================================

import os
import sys
import gc
import torch

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)

# Clear old project modules
for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(42)

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from src.models.unet import MagnitudeUNet, UNetConfig
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ------------------------------------------------------------
# Configs
# ------------------------------------------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

# Final fine-tuning settings
new_learning_rate = 1e-4
more_epochs = 10
log_every = 50

# ------------------------------------------------------------
# Build model and loaders
# ------------------------------------------------------------
stft_processor = STFTProcessor(stft_config).to(device)

model = MagnitudeUNet(model_config).to(device)

loss_function = Phase1Loss(
    w_l1=0.7,
    w_sisnr=0.3,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=new_learning_rate,
    weight_decay=1e-4,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
checkpoint_path = os.path.join(
    project_root,
    "checkpoints",
    "phase1",
    "best_model.pt",
)

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])

# IMPORTANT:
# We do NOT load optimizer state because we want to force LR = 1e-4

history = checkpoint["history"]
start_epoch = checkpoint["epoch"] + 1

# Best SI-SDR so far
if "val_si_sdr" in history and len(history["val_si_sdr"]) > 0:
    best_val_sdr = max(history["val_si_sdr"])
else:
    best_val_sdr = checkpoint.get("val_si_sdr", float("-inf"))

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Resuming from epoch {start_epoch - 1}")
print(f"Best val SI-SDR so far: {best_val_sdr:+.2f} dB")
print(f"New learning rate: {new_learning_rate}")
print(f"Running epochs: {start_epoch} to {start_epoch + more_epochs - 1}")
print()

# ------------------------------------------------------------
# Helper: compute SI-SDR on CPU to save GPU memory
# ------------------------------------------------------------
def cpu_si_sdr(pred_audio: torch.Tensor, target_audio: torch.Tensor) -> float:
    return compute_si_sdr(
        pred_audio.detach().cpu(),
        target_audio.detach().cpu(),
    )

# ============================================================
# Training loop
# ============================================================
for epoch in range(start_epoch, start_epoch + more_epochs):

    # -------------------------
    # Train
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_sdr_sum = 0.0
    train_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio = target_audio.to(device)

        # STFT
        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        # Predict masks for left/right channels
        predicted_masks = []
        for ch in range(mixture_mag.shape[1]):
            one_channel_mag = mixture_mag[:, ch:ch+1, :, :]
            one_channel_mask = model(one_channel_mag)
            predicted_masks.append(one_channel_mask)

        predicted_mask = torch.cat(predicted_masks, dim=1)
        predicted_mag = mixture_mag * predicted_mask

        # Convert predicted magnitude back to waveform
        predicted_stft = stft_processor.magnitude_phase_to_complex(
            predicted_mag,
            mixture_phase,
        )
        predicted_audio = stft_processor.istft(
            predicted_stft,
            length=original_length,
        )

        # Loss
        loss_dict = loss_function(
            predicted_mag,
            target_mag,
            predicted_audio,
            target_audio,
        )
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        # Log metric every few batches
        if batch_idx % log_every == 0:
            current_sdr = cpu_si_sdr(predicted_audio, target_audio)
            train_sdr_sum += current_sdr
            train_sdr_count += 1

            avg_loss_so_far = train_loss_sum / (batch_idx + 1)
            avg_sdr_so_far = train_sdr_sum / train_sdr_count

            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss_so_far:.4f} | SI-SDR: {avg_sdr_so_far:+.2f} dB"
            )

        # Cleanup
        del mixture_audio
        del target_audio
        del mixture_mag
        del mixture_phase
        del target_mag
        del predicted_mask
        del predicted_mag
        del predicted_stft
        del predicted_audio
        del loss_dict
        del total_loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr = train_sdr_sum / train_sdr_count if train_sdr_count > 0 else 0.0

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_sdr_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            predicted_masks = []
            for ch in range(mixture_mag.shape[1]):
                one_channel_mag = mixture_mag[:, ch:ch+1, :, :]
                one_channel_mask = model(one_channel_mag)
                predicted_masks.append(one_channel_mask)

            predicted_mask = torch.cat(predicted_masks, dim=1)
            predicted_mag = mixture_mag * predicted_mask

            predicted_stft = stft_processor.magnitude_phase_to_complex(
                predicted_mag,
                mixture_phase,
            )
            predicted_audio = stft_processor.istft(
                predicted_stft,
                length=original_length,
            )

            loss_dict = loss_function(
                predicted_mag,
                target_mag,
                predicted_audio,
                target_audio,
            )

            current_val_sdr = cpu_si_sdr(predicted_audio, target_audio)

            val_loss_sum += loss_dict["loss"].item()
            val_sdr_sum += current_val_sdr
            val_count += 1

            del mixture_audio
            del target_audio
            del mixture_mag
            del mixture_phase
            del target_mag
            del predicted_mask
            del predicted_mag
            del predicted_stft
            del predicted_audio
            del loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr = val_sdr_sum / val_count

    # -------------------------
    # Save history
    # -------------------------
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    current_best_sdr = max(history["val_si_sdr"])

    # -------------------------
    # Epoch summary
    # -------------------------
    print("\n" + "=" * 55)
    print(f"Epoch {epoch} done")
    print(f"  Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f"  Train SDR  : {avg_train_sdr:+.2f} dB | Val SDR  : {avg_val_sdr:+.2f} dB")
    print(f"  Best so far: {current_best_sdr:+.2f} dB | Target   : +3.00 dB")
    print("=" * 55 + "\n")

    # -------------------------
    # Save if SI-SDR improves
    # -------------------------
    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": avg_val_loss,
                "val_si_sdr": avg_val_sdr,
                "history": history,
            },
            checkpoint_path,
        )

        print(f"★ New best checkpoint saved! Val SI-SDR = {avg_val_sdr:+.2f} dB\n")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
best_sdr = max(history["val_si_sdr"])

print("=" * 55)
print(f"FINAL BEST VAL SI-SDR: {best_sdr:+.2f} dB")
if best_sdr >= 3.0:
    print(" PHASE 1 COMPLETE! We can move to Phase 2.")
else:
    print(f"Still {3.0 - best_sdr:.2f} dB away.")
    print("accept this Phase 1 baseline.")
print("=" * 55)

Device: cuda
MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loaded checkpoint: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase1\best_model.pt
Resuming from epoch 26
Best val SI-SDR so far: +2.79 dB
New learning rate: 0.0001
Running epochs: 27 to 36

Epoch 27 | Batch 000/500 | Loss: -0.6955 | SI-SDR: +2.82 dB
Epoch 27 | Batch 050/500 | Loss: -0.2954 | SI-SDR: +3.40 dB
Epoch 27 | Batch 100/500 | Loss: -0.2812 | SI-SDR: +2.95 dB
Epoch 27 | Batch 150/500 | Loss: -0.3149 | SI-SDR: +3.62 dB
Epoch 27 | Batch 200/500 | Loss: -0.3980 | SI-SDR: +3.28 dB
Epoch 27 | Batch 250/500 | Loss: -0.3971 | SI-SDR: +3.39 dB
Epoch 27 | Batc

In [11]:
import os
import shutil

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

# Source: current best Phase 1 model
phase1_best = os.path.join(project_root, "checkpoints", "phase1", "best_model.pt")

# Destination: permanently saved Phase 1 result
phase1_final = os.path.join(project_root, "checkpoints", "phase1", "phase1_final_3.22dB.pt")

if os.path.exists(phase1_best):
    shutil.copy(phase1_best, phase1_final)
    print(f"Phase 1 final checkpoint saved:")
    print(f"  {phase1_final}")
    print(f"  Size: {os.path.getsize(phase1_final) / 1e6:.1f} MB")
else:
    print("Could not find best_model.pt — check the path")

print()
print("Phase 1 checkpoint is safe.")


Phase 1 final checkpoint saved:
  C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase1\phase1_final_3.22dB.pt
  Size: 40.3 MB

Phase 1 checkpoint is safe.
